# Beamer-Minor Method (BMM) Regression Analysis
Python/Google Earth Engine (GEE) workflow to linearly regress eddy covariance (EC) and bowen ratio (BR) flux station measurements of actual ET (normalized by climate data) with Landsat-derived vegetation indices (VI)

## Workflow:
1. Energy balance closure corrections for flux data<br>
   a. Export daily gridMET ETo and Precip for eddy covariance flux stations<br>
   b. Export daily gridMET ETo and Precip for bown ratio flux stations<br>
   c. Visual inspection and automated QC flagging of flux station data<br>
   d. Correct fluxes and gap-fill using [flux-data-qaqc](https://flux-data-qaqc.readthedocs.io/en/latest/)<br>
   e. Calculate annual corrected actual ET (ETa) totals for all flux stations
---
2. Vegetation index processing<br>
   a. Develop Windrose diagram to help determine which way to shift flux footprint/source area polygons<br>
   b. Export spatially-averaged vegetation index values for buffers around flux stations  (i.e., source areas/flux footprints)<br>
   c. Join vegetation index summary with flux data summary for regression analysis<br>
---
3. Regression analysis

In [1]:

#####################################################################

# specify root path for the local system path to the github repo
root_path = r'E:\nwi-t2-gw-discharge'

# specify google cloud project ID for GEE exports (only used for step 2b.)
gcloud_project_id = 'ee-bminor'

#####################################################################

import os

if os.path.isdir(root_path):
    print('root directory was found, proceed')
else:
    print('root directory for the local copy of the github repo was not found, make sure the root_path variable above is defined correctly')

root directory was found, proceed


### Import modules and initialize GEE Python API

In [2]:
import ee
import os

import pandas as pd
import numpy as np
import datetime as dt
from glob import glob

# stats
from rosely import WindRose
from numpy.polynomial import polynomial as poly
from scipy.optimize import curve_fit
import scipy.stats
from scipy.stats.distributions import t
from sklearn import linear_model as lm
import statsmodels.api as sm
import uncertainties as unc
import uncertainties.unumpy as unp
from math import pi

# plotting
from bokeh.models import ColumnDataSource, RangeTool, HoverTool, LinearAxis, Range1d
from bokeh.layouts import column
from bokeh.plotting import figure, show
from bokeh.palettes import Sunset9
from bokeh.plotting import figure, show, output_file, reset_output, ColumnDataSource
from bokeh.models import HoverTool, CDSView, IndexFilter, Whisker, Ticker, TickFormatter, FactorRange, ColorBar
from bokeh.models.annotations import Label
from bokeh.io import output_notebook, export_png
from bokeh.transform import dodge, linear_cmap
import matplotlib.pyplot as plt

ee.Initialize(project=gcloud_project_id)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_7TDKVSyKvBdmMqW?ref=4i2o6


## 1. Energy balance closure corrections for flux data
---

### 1a. Export daily gridMET ETo and PPT for eddy covariance flux stations

In [ ]:

table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')

in_path = os.path.join(table_path, 'eddy_covariance_station_data')

# bowen ratio sites to exclude
filt_list = ['ET_1', 'ET_8', 'B_01', 'B_11', 'MOVAL', 'MR', 'VR', 'TAM', 'UOVLO', 'UOVMD', 'UOVUP', 'CC', 'EGBRX', 'PKL']

# list of gridMET and Daymet data variable band names
data_vars_lst_g = ['etr', 'eto', 'pr']
data_vars_out_lst_g = ['gridMET_ETr', 'gridMET_ETo', 'gridMET_prcp']

data_vars_lst_d = ['prcp']
data_vars_out_lst_d = ['daymet_prcp']

######################################################################

# OpenET Reference ET station Master List
try:
    stn_df = pd.read_csv(os.path.join(table_path, 'site_year_list.csv'))
    stn_df = stn_df[~stn_df['Site'].duplicated(keep='first')]
except Exception as e:
    print(e)


# loop through each row of specified dataframe
for row in stn_df.itertuples():

    stn_nm = row[1]

    stn_nm = stn_nm.replace('-', '_')

    if str(stn_nm) in filt_list:
        continue
    
    print(f'Processing station {stn_nm}')

    # station lat/long
    lat = row[5]
    lon = row[6]

    if (stn_nm == 'ERG_2' or stn_nm == 'LONS'):
        try:
            flux_df = pd.read_excel(os.path.join(in_path, f'raw_daily_{stn_nm}.xlsx'), index_col=0, parse_dates=True)
        except Exception as e:
            print(e)
    else:
        try:
            flux_df = pd.read_excel(os.path.join(in_path, f'raw_subhour_{stn_nm}.xlsx'), index_col=0, parse_dates=True)
        except Exception as e:
            print(e)

    flux_df = flux_df.dropna(axis=0, how='all')
    
    flux_df.index = pd.to_datetime(flux_df.index, format='mixed')
    
    # data collection period
    dat_start = flux_df.index[0].date()
    dat_end = flux_df.index[-1].date()
    
    # advance one day past end date since EE end date is exclusive not inclusive
    dat_end_ee = dat_end + dt.timedelta(days=1)

    # format datetimes as strings without the time portion
    dat_start_str = f'{dat_start:%Y-%m-%d}'
    dat_end_str = f'{dat_end:%Y-%m-%d}'

    dat_end_str_ee = f'{dat_end_ee:%Y-%m-%d}'

    
    # Earth Engine Point Geometry 
    stn_pnt = ee.Geometry.Point(lon, lat)


    # prep gridmet image collection
    grid_coll = (
        ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
            .select(data_vars_lst_g, data_vars_out_lst_g)
            .filter(ee.Filter.date(dat_start_str, dat_end_str_ee))
    )

    # Daymet missing 2012-12-31
    # daym_coll = (
    #     ee.ImageCollection("NASA/ORNL/DAYMET_V4")
    #         .select(data_vars_lst_d, data_vars_out_lst_d)
    #         .filter(ee.Filter.date(dat_start_str, dat_end_str_ee))
    # )

    # # join image collections on dates using a simple inner join
    # filter_R = ee.Filter.equals(leftField='system:index', rightField='system:index')

    # simpleJoinR = ee.Join.inner()

    # innerJoinR = ee.ImageCollection(simpleJoinR.apply(grid_coll, daym_coll,  filter_R))

    # def joinFunc(img):
    #     return ee.Image.cat(img.get('primary'), img.get('secondary'))

    # # final image collection to be used for zonal stats
    # coll = innerJoinR.map(joinFunc)
    

    # zonal stats function to get daily values extracted for station
    def zonalStatsFunc(img):
        img = img.set({  
                    'date': ee.Date(img.get('system:time_start')).format('yyyy-MM-dd'),
                    # 'year': ee.Date(img.get('system:time_start')).get('year'),
                    # 'month': ee.Date(img.get('system:time_start')).get('month'),
                    # 'day': ee.Date(img.get('system:time_start')).get('day'),
            })

        stats_info = (
            img
                .reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=stn_pnt
                )
        )

        # add the image properties to the output
        stats_info = stats_info.combine(img.toDictionary())

        return ee.Feature(None, stats_info)

    stats_coll = ee.FeatureCollection(
            grid_coll.map(zonalStatsFunc))

    # concatenate variable list to additional properties for the output field selection
    # fin_lst = ['date'] + data_vars_out_lst_g + data_vars_out_lst_d
    fin_lst = ['date'] + data_vars_out_lst_g

    # output filename
    exp_filename = f'{stn_nm}_gridmet_daily'

    # Export a CSV file to Cloud Storage.
    out_data_task = ee.batch.Export.table.toDrive(**{
      'collection': stats_coll,
      'description': f'gridMETToDrive_{stn_nm}',
      'folder': 'phreatophyte_sites_gridmet_data',
      'fileNamePrefix': exp_filename,
      'fileFormat': 'CSV',
      'selectors': fin_lst
    })

    # out_data_task.start()

print('finished downloading gridMET/Daymet daily data for eddy covariance sites')

### 1b. Export daily gridMET ETo and PPT for bowen ratio flux stations

In [ ]:

table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')

in_path = os.path.join(table_path, 'bowen_ratio_station_data')

# bowen ratio sites
filt_list = ['ET_1', 'ET_8', 'B_01', 'B_11', 'MR', 'VR', 'TAM', 'UOVLO', 'UOVMD', 'UOVUP', 'MOVAL', 'CC', 'EGBRX', 'PKL']

# list of gridmet and dayment data variable band names
data_vars_lst_g = ['etr', 'eto', 'pr']
data_vars_out_lst_g = ['gridMET_ETr', 'gridMET_ETo', 'gridMET_prcp']

data_vars_lst_d = ['prcp']
data_vars_out_lst_d = ['daymet_prcp']

######################################################################

# OpenET Reference ET station Master List
try:
    stn_df = pd.read_csv(os.path.join(table_path, 'site_year_list.csv')
    stn_df = stn_df[~stn_df['Site'].duplicated(keep='first')]
except Exception as e:
    print(e)

# loop through each row of specified dataframe
for row in stn_df.itertuples():

    stn_nm = row[1]

    stn_nm = stn_nm.replace('-', '_')

    if str(stn_nm) not in filt_list:
        continue

    print(f'Processing station {stn_nm}')

    # station lat/long
    lat = row[5]
    lon = row[6]

    if (stn_nm == 'MOVAL' or stn_nm == 'UOVLO' or stn_nm == 'UOVMD' or stn_nm == 'UOVUP' or stn_nm == 'ET_1' or stn_nm == 'CC' or stn_nm == 'EGBRX' or stn_nm == 'PKL'):
        try:
            flux_df = pd.read_excel(os.path.join(in_path, f'raw_daily_{stn_nm}.xlsx'), index_col=0, parse_dates=True)
        except Exception as e:
            print(e)
    else:
        try:
            flux_df = pd.read_excel(os.path.join(in_path, f'raw_subhour_{stn_nm}.xlsx'), index_col=0, parse_dates=True)
        except Exception as e:
            print(e)

    flux_df = flux_df.dropna(axis=0, how='all')
    
    flux_df.index = pd.to_datetime(flux_df.index, format='mixed')
    
    # data collection period
    dat_start = flux_df.index[0].date()
    dat_end = flux_df.index[-1].date()
    
    # advance one day past end date since EE end date is exclusive not inclusive
    dat_end_ee = dat_end + dt.timedelta(days=1)

    # format datetimes as strings without the time portion
    dat_start_str = f'{dat_start:%Y-%m-%d}'
    dat_end_str = f'{dat_end:%Y-%m-%d}'

    dat_end_str_ee = f'{dat_end_ee:%Y-%m-%d}'
    
    
    # Earth Engine Point Geometry 
    stn_pnt = ee.Geometry.Point(lon, lat)


    # prep gridmet image collection
    grid_coll = (
        ee.ImageCollection("IDAHO_EPSCOR/GRIDMET")
            .select(data_vars_lst_g, data_vars_out_lst_g)
            .filter(ee.Filter.date(dat_start_str, dat_end_str_ee))
    )


    # daym_coll = (
    #     ee.ImageCollection("NASA/ORNL/DAYMET_V4")
    #         .select(data_vars_lst_d, data_vars_out_lst_d)
    #         .filter(ee.Filter.date(dat_start_str, dat_end_str_ee))
    # )


    # join image collections on dates using a simple inner join
    # filter_R = ee.Filter.equals(leftField='system:index', rightField='system:index')

    # simpleJoinR = ee.Join.inner()

    # innerJoinR = ee.ImageCollection(simpleJoinR.apply(grid_coll, daym_coll,  filter_R))

    # def joinFunc(img):
    #     return ee.Image.cat(img.get('primary'), img.get('secondary'))

    # # final image collection to be used for zonal stats
    # coll = innerJoinR.map(joinFunc)
    

    # zonal stats function to get daily values extracted for station
    def zonalStatsFunc(img):
        img = img.set({  
                    'date': ee.Date(img.get('system:time_start')).format('yyyy-MM-dd'),
                    # 'year': ee.Date(img.get('system:time_start')).get('year'),
                    # 'month': ee.Date(img.get('system:time_start')).get('month'),
                    # 'day': ee.Date(img.get('system:time_start')).get('day'),
            })

        stats_info = (
            img
                .reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=stn_pnt
                )
        )

        # add the image properties to the output
        stats_info = stats_info.combine(img.toDictionary())

        return ee.Feature(None, stats_info)

    stats_coll = ee.FeatureCollection(
            grid_coll.map(zonalStatsFunc))

    # concatenate variable list to additional properties for the output field selection
    # fin_lst = ['date'] + data_vars_out_lst_g + data_vars_out_lst_d
    fin_lst = ['date'] + data_vars_out_lst_g

    # output filename
    exp_filename = f'{stn_nm}_gridmet_daily'

    # Export a CSV file to Cloud Storage.
    out_data_task = ee.batch.Export.table.toDrive(**{
      'collection': stats_coll,
      'description': f'gridMETToDrive_{stn_nm}',
      'folder': 'phreatophyte_sites_gridmet_data',
      'fileNamePrefix': exp_filename,
      'fileFormat': 'CSV',
      'selectors': fin_lst
    })

    # out_data_task.start()

    
print('finished downloading gridMET/Daymet daily data for bowen ratio sites')
 

### 1c. Visual inspection and automated QC flagging of flux station data:
#### Law et al. (2005) AmeriFlux guidelines (https://citeseerx.ist.psu.edu/document?repid=rep1&type=pdf&doi=25952cbc1526c1916a0652d2b098e0a6e6791d52)
#### Garcia et al. (2016) (https://pubs.usgs.gov/pp/1805/pdf/pp1805.pdf)
> Sensible heat flux (H), W/m2, -150/700<br>
> Latent heat flux (LE), W/m2, -50/700 when Rn > 5 and -50/50 when Rn < 5<br>
> Net radiation (Rn), W/m2, -200/900<br>
> Soil heat flux (G), W/m2, -50/50 (using -200/200 based on measured values from phreatophyte sites (e.g., AFD site) and strictness of -50/50 range)<br>


In [ ]:
#########################################################################

# eddy covariance datasets (EC) or bowen ratio datasets (BR), files are stored in separate sub-folders
station_data_type = 'BR'

# flag to plot the timeseries for visual inspection
plotting_flag = True

#########################################################################

In [ ]:


table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')


if station_data_type == 'EC':
    in_path = os.path.join(table_path, 'eddy_covariance_station_data')
elif station_data_type == 'BR':
    in_path = os.path.join(table_path, 'bowen_ratio_station_data')

# list of all files within the input folder
all_files = os.listdir(in_path)

# list of site IDs within the input folder
site_list = [f.split('.')[0].replace('raw_subhour_', '').replace('raw_daily_', '') for f in all_files if f.endswith('csv')]
# site_list = ['ET_1']

for site in site_list:

    print(f'processing {site}')

    try:
        if (site == 'ERG_2' or site == 'LONS' or site == 'CC' or site == 'EGBRX' or site == 'ET_1' or site == 'MOVAL' or site == 'PKL' or site == 'UOVLO' or site == 'UOVMD' or site == 'UOVUP'):
            df = pd.read_excel(os.path.join(in_path, f'raw_daily_{site}.xlsx'), index_col=0, parse_dates=True, na_values='NO_DATA')
        else:
            df = pd.read_excel(os.path.join(in_path, f'raw_subhour_{site}.xlsx'), index_col=0, parse_dates=True, na_values='NO_DATA')
    except Exception as e:
        print(e)

    # make QC flag columns
    if (site == 'AWSR_S' or site == 'AWSR_W'):
        df['H (W/m2) QC'] = 1.0
        df['LE (W/m2) QC'] = 1.0
        df['Rn (W/m2) QC'] = 1.0
        df['G (W/m2) QC'] = 1.0
        
        data_le_length = len(df['LE (W/m2)'])
        data_h_length = len(df['H (W/m2)'])
        data_rn_length = len(df['Rn (W/m2)'])
        data_g_length = len(df['G (W/m2)'])
    else:
        df['Sensible-heat flux W/m2 QC'] = 1.0
        df['Latent-heat flux W/m2 QC'] = 1.0
        df['Net radiation W/m2 QC'] = 1.0
        df['Soil-heat flux W/m2 QC'] = 1.0

        data_le_length = len(df['Latent-heat flux W/m2'])
        data_h_length = len(df['Sensible-heat flux W/m2'])
        data_rn_length = len(df['Net radiation W/m2'])
        data_g_length = len(df['Soil-heat flux W/m2'])
    
    # mark poor quality data with lower QC values
    if (site == 'AWSR_S' or site == 'AWSR_W'):
        df.loc[(df['H (W/m2)'] > 700) | (df['H (W/m2)'] < -150), 'H (W/m2) QC'] = 0.5
        df.loc[(df['LE (W/m2)'] > 700) & (df['Rn (W/m2)'] > 5), 'LE (W/m2) QC'] = 0.5
        df.loc[(df['LE (W/m2)'] < -50) & (df['Rn (W/m2)'] > 5), 'LE (W/m2) QC'] = 0.5
        df.loc[(df['LE (W/m2)'] > 50) & (df['Rn (W/m2)'] < 5), 'LE (W/m2) QC'] = 0.5
        df.loc[(df['LE (W/m2)'] < -50) & (df['Rn (W/m2)'] < 5), 'LE (W/m2) QC'] = 0.5
        df.loc[(df['Rn (W/m2)'] > 900) | (df['Rn (W/m2)'] < -200), 'Rn (W/m2) QC'] = 0.5
        df.loc[(df['G (W/m2)'] > 200) | (df['G (W/m2)'] < -200), 'G (W/m2) QC'] = 0.5

        # get % of total data flagged as poor quality
        le_length = len(df.loc[df['LE (W/m2) QC'] < 1])
        h_length = len(df.loc[df['H (W/m2) QC'] < 1])
        rn_length = len(df.loc[df['Rn (W/m2) QC'] < 1])
        g_length = len(df.loc[df['G (W/m2) QC'] < 1])
    else:
        df.loc[(df['Sensible-heat flux W/m2'] > 700) | (df['Sensible-heat flux W/m2'] < -150), 'Sensible-heat flux W/m2 QC'] = 0.5
        df.loc[(df['Latent-heat flux W/m2'] > 700) & (df['Net radiation W/m2'] > 5), 'Latent-heat flux W/m2 QC'] = 0.5
        df.loc[(df['Latent-heat flux W/m2'] < -50) & (df['Net radiation W/m2'] > 5), 'Latent-heat flux W/m2 QC'] = 0.5
        df.loc[(df['Latent-heat flux W/m2'] > 50) & (df['Net radiation W/m2'] < 5), 'Latent-heat flux W/m2 QC'] = 0.5
        df.loc[(df['Latent-heat flux W/m2'] < -50) & (df['Net radiation W/m2'] < 5), 'Latent-heat flux W/m2 QC'] = 0.5
        df.loc[(df['Net radiation W/m2'] > 900) | (df['Net radiation W/m2'] < -200), 'Net radiation W/m2 QC'] = 0.5
        df.loc[(df['Soil-heat flux W/m2'] > 200) | (df['Soil-heat flux W/m2'] < -200), 'Soil-heat flux W/m2 QC'] = 0.5

        # get % of total data flagged as poor quality
        le_length = len(df.loc[df['Latent-heat flux W/m2 QC'] < 1])
        h_length = len(df.loc[df['Sensible-heat flux W/m2 QC'] < 1])
        rn_length = len(df.loc[df['Net radiation W/m2 QC'] < 1])
        g_length = len(df.loc[df['Soil-heat flux W/m2 QC'] < 1])
    
    le_per = round((le_length / data_le_length) * 100, 3)
    h_per = round((h_length / data_h_length) * 100, 3)
    rn_per = round((rn_length / data_rn_length) * 100, 3)
    g_per = round((g_length / data_g_length) * 100, 3)
    print('Bad QC data lengths out of total:')
    print(f'% of LE: {le_per}')
    print(f'% of H: {h_per}')
    print(f'% of Rn: {rn_per}')
    print(f'% of G: {g_per}')
        
    
    if plotting_flag:
        dates = np.array(df.index, dtype=np.datetime64)
        if (site == 'AWSR_S' or site == 'AWSR_W'):
            source = ColumnDataSource(data=dict(date=dates, 
                                                lat=df['LE (W/m2)'], sen=df['H (W/m2)'], rad=df['Rn (W/m2)'], soi=df['G (W/m2)'],
                                                latq=df['LE (W/m2) QC'], senq=df['H (W/m2) QC'], radq=df['Rn (W/m2) QC'], soiq=df['G (W/m2) QC']))
        else:
            source = ColumnDataSource(data=dict(date=dates, 
                                                lat=df['Latent-heat flux W/m2'], sen=df['Sensible-heat flux W/m2'], rad=df['Net radiation W/m2'], soi=df['Soil-heat flux W/m2'],
                                                latq=df['Latent-heat flux W/m2 QC'], senq=df['Sensible-heat flux W/m2 QC'], radq=df['Net radiation W/m2 QC'], soiq=df['Soil-heat flux W/m2 QC']))

        if (site == 'ERG_2' or site == 'LONS' or site == 'CC' or site == 'EGBRX' or site == 'ET_1' or site == 'MOVAL' or site == 'PKL' or site == 'UOVLO' or site == 'UOVMD' or site == 'UOVUP'):
            p = figure(height=450, width=1300, tools="xpan", toolbar_location=None,
                       x_axis_type="datetime", x_axis_location="above", title=f"{site} Flux Station Timeseries",
                       background_fill_color="#efefef", x_range=(dates[50], dates[100]))
        else:
            p = figure(height=450, width=1300, tools="xpan", toolbar_location=None,
                           x_axis_type="datetime", x_axis_location="above", title=f"{site} Flux Station Timeseries",
                           background_fill_color="#efefef", x_range=(dates[1500], dates[2500]))

        p.line(x='date', y='lat', source=source, line_color='blue', legend_label="LE")
        p.line(x='date', y='sen', source=source, line_color='red', legend_label="H")
        p.line(x='date', y='rad', source=source, line_color='black', legend_label="Rn")
        p.line(x='date', y='soi', source=source, line_color='green', legend_label="G")
        p.yaxis.axis_label = 'W/m2'
        
        # Setting the second y axis range name and range
        p.extra_y_ranges = {"qc": Range1d(start=0, end=1.2)}

        # Adding the second axis to the plot.  
        p.add_layout(LinearAxis(y_range_name="qc", axis_label='QC value'), 'right')
        p.scatter(x='date', y='latq', source=source, marker='circle', color='blue', y_range_name='qc', legend_label='QC LE')
        p.scatter(x='date', y='senq', source=source, marker='circle', color='red', y_range_name='qc', legend_label='QC H')
        p.scatter(x='date', y='radq', source=source, marker='circle', color='black', y_range_name='qc', legend_label='QC Rn')
        p.scatter(x='date', y='soiq', source=source, marker='circle', color='green', y_range_name='qc', legend_label='QC G')
        p.legend.location = "bottom_right"
        p.legend.click_policy ="hide"

        hover = HoverTool(
                        tooltips=[
                            ("date", "$x{%F %H:%M}"),
                            ("LE", "@lat{0.00}"),
                            ("H", "@sen{0.00}"),
                            ("Rn", "@rad{0.00}"),
                            ("G", "@soi{0.00}")],
                        formatters={'$x': 'datetime'}
                    )

        # add hover tool to the plot
        p.add_tools(hover)
        
        select = figure(title="Drag the middle and edges of the selection box to change the range above",
                        height=300, width=1300, y_range=p.y_range,
                        x_axis_type="datetime", y_axis_type=None,
                        tools="", toolbar_location=None, background_fill_color="#efefef")
        
        range_tool = RangeTool(x_range=p.x_range, start_gesture="pan")
        range_tool.overlay.fill_color = "navy"
        range_tool.overlay.fill_alpha = 0.2
        
        select.line(x='date', y='lat', source=source, line_color='blue', legend_label="LE")
        select.line(x='date', y='sen', source=source, line_color='red', legend_label="H")
        select.line(x='date', y='rad', source=source, line_color='black', legend_label="Rn")
        select.line(x='date', y='soi', source=source, line_color='green', legend_label="G")
        select.ygrid.grid_line_color = None
        select.legend.click_policy="hide"
        select.legend.location = "bottom_right"
        select.yaxis.axis_label = 'W/m2'
        select.add_tools(range_tool)
        
        show(column(p, select))

    df = df.reset_index()
    
    # if (site == 'ERG_2' or site == 'LONS' or site == 'CC' or site == 'EGBRX' or site == 'ET_1' or site == 'MOVAL' or site == 'PKL' or site == 'UOVLO' or site == 'UOVMD' or site == 'UOVUP'):
    #     df.to_csv(os.path.join(in_path, f'raw_daily_{site}.csv'), index=False, na_rep='NO_DATA')
    # else:
    #     df.to_csv(os.path.join(in_path, f'raw_subhour_{site}.csv'), index=False, na_rep='NO_DATA')

    print(f'processed {site}')

### 1d. EBC corrections of eddy covariance and gap filling for eddy covariance/bowen ratio flux station data

#### Run the two code blocks below from the Command Line/Terminal
> One code block is for processing eddy covariance flux data and the other is for processing bowen ratio flux data; different parameters/methods of processing<br>
> **NOTE**: Can only process one site at a time during command line/terminal session<br><br>
> Need to first activate the ***fluxdataqaqc*** python environment after following setup instructions [here](https://flux-data-qaqc.readthedocs.io/en/latest/install.html)<br>
> Then navigate to the local system's location of the ***flux-data-qaqc*** github repository after downloading/cloning from [here](https://github.com/Open-ET/flux-data-qaqc)<br>
> Ensure that the configuration file (e.g., config_AFD.ini) for the flux station/site_id is prepared prior to running the blocks below 

#### Example navigation code below:<br>

    # activate the environment
    conda activate fluxdataqaqc

    # use "change directory", cd for short, to navigate to the local copy of the flux-data-qaqc github repository
    cd E:\flux-data-qaqc

    # initiate a python session
    python


#### Eddy Covariance data:

    # import modules
    import pandas as pd
    from fluxdataqaqc import Data, QaQc, Plot
    from bokeh.plotting import figure, show, ColumnDataSource
    from bokeh.models.formatters import DatetimeTickFormatter
    from bokeh.models import LinearAxis, Range1d
    from bokeh.io import output_notebook

    # define the site to process
    site_id = "AFD"

    # define the local path to the location where the station configuration file is stored
    config_path = fr"E:\nwi-t2-gw-discharge\tables\regression_analysis\eddy_covariance_station_data\config_{site_id}.ini"

    # create the Data object using the config file
    d = Data(config_path)

    # check to make sure QC columns are recognized
    d.qc_var_pairs

    # apply the QC flags to the data
    d.apply_qc_flags()

    # plot the original timeseries
    d.plot(plot_width=1100, plot_height=600)

    # create the QaQc object
    q = QaQc(d, daily_frac=20/24, max_interp_hours=2, max_interp_hours_night=2, drop_gaps=False)

    # correct the data using the energy budget ratio method and gap-fill
    q.correct_data(meth='ebr', et_gap_fill=True, refET='ETo')

    # write the corrected data to file
    q.write()

    # plot the corrected data
    q.plot(plot_width=1100, plot_height=600)


---

#### Bowen Ratio data:

    # import modules
    import pandas as pd
    from fluxdataqaqc import Data, QaQc, Plot
    from bokeh.plotting import figure, show, ColumnDataSource
    from bokeh.models.formatters import DatetimeTickFormatter
    from bokeh.models import LinearAxis, Range1d
    from bokeh.io import output_notebook

    # define the site to process
    site_id = "MOVAL"

    # define the local path tothe  lcoation where the station configuration file is stored
    config_path = fr"E:\MinorBeamer\2024_update\bowen_ratio_station_data\config_{site_id}.ini"

    # create the Data object using the config file
    d = Data(config_path)

    # check to make sure QC columns are recognized
    d.qc_var_pairs

    # apply the QC flags to the data
    d.apply_qc_flags()
    
    # plot the original timeseries
    d.plot(plot_width=1100, plot_height=600)

    # plot the QaQc object
    q = QaQc(d, daily_frac=20/24, max_interp_hours=2, max_interp_hours_night=2, drop_gaps=False)

    # correct the data using the bowen ratio method and gap-fill
    q.correct_data(meth='br', et_gap_fill=True, refET='ETo')

    # write the corrected data to file
    q.write()

    # plot the corrected data
    q.plot(plot_width=1100, plot_height=600)

---

### 1e. Calculate annual corrected actual ET (ETa) totals for all flux stations
> Requires corrected flux datasets for all sites to be processed from the previous step

In [12]:
table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')

# read the site-year list with respective station data start/end dates
try:
    m_list_df = pd.read_csv(os.path.join(table_path, 'site_year_list.csv'))
except Exception as e:
    print(e)

m_list_df = m_list_df[['Site', 'Reference', 'Year', 'STATION_METHOD', 'START_DATE', 'END_DATE']]

# empty columns that will be filled
m_list_df['Initial EB Slope'] = np.nan
m_list_df['Corrected EB Slope'] = np.nan
m_list_df['ETa (mm/yr)'] = np.nan

# loop through all site-years
for row_tuple in m_list_df.itertuples():
    
    site_id = row_tuple[1]
    site_id_alt = site_id.replace('-', '_')

    year = row_tuple[3]

    station_typ = row_tuple[4]

    start_dt_str = pd.to_datetime(row_tuple[5]).strftime('%Y-%m-%d')
    end_dt_str = pd.to_datetime(row_tuple[6]).strftime('%Y-%m-%d')

    if station_typ == 'EC':
        in_path_pre = os.path.join(table_path, 'eddy_covariance_station_data')
    elif station_typ == 'BR':
        in_path_pre = os.path.join(table_path, 'bowen_ratio_station_data')
    
    in_path = os.path.join(in_path_pre, 'output')

    try:
        # read the daily data file
        stn_timeseries_df = pd.read_csv(os.path.join(in_path, f'{site_id_alt}_daily_data.csv'), usecols=['date', 'ET_corr', 'flux', 'energy', 'flux_corr'], index_col=0, parse_dates=True)

        # EC site with special handling (i.e., substituted data from the MR bowen ratio site due to instrumentation failure)
        if site_id == 'LMVW':

            # filter timeseries dataframe to the current station's water year
            df1 = stn_timeseries_df.loc['2006-04-12': end_dt_str]
    
            # merging MR gap-filled data into LMVW time series for full water year ETa
            try:
                mr_file_path_pre = os.path.join(table_path, 'bowen_ratio_station_data')
                mr_file_path = os.path.join(mr_file_path_pre, 'output')
                
                df2 = pd.read_csv(os.path.join(mr_file_path, 'MR_daily_data.csv'), usecols=['date', 'ET_corr', 'flux', 'energy', 'flux_corr'], index_col=0, parse_dates=True)
                
            except Exception as e:
                print(e)
                print('EBC-corrected data for MR not found, which is a requirement for the LMVW site')

            df2 = df2.loc[start_dt_str: '2006-04-11']
            
            df1 = pd.concat([df2, df1], axis=0)

            df1_e = df1[['energy', 'flux', 'flux_corr']].dropna(how='any')
            
            coeffs_ini, stats_ini = poly.polyfit(df1_e['energy'], df1_e['flux'], 1, full=True)
            coeffs_cor, stats_cor = poly.polyfit(df1_e['energy'], df1_e['flux_corr'], 1, full=True)
            m_ini = coeffs_ini[1].round(3)
            m_cor = coeffs_cor[1].round(3)
            
            # get annual sum of ETa
            df1_s = df1['ET_corr'].sum().round(2)

            # populate the output dataframe with the stats
            m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Initial EB Slope'] = m_ini
            m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Corrected EB Slope'] = m_cor

        # BR sites with special handling (i.e., using reported values from DeMeo et al. (2008))
        # elif (site_id == 'MR' or site_id == 'VR'):

        #     # filter timeseries dataframe to the current station's water year
        #     df1 = stn_timeseries_df.loc[start_dt_str: end_dt_str]
            
        #     # calculate DOY average ET and then sum up the annual total
        #     # Extract day of year
        #     stn_timeseries_df['day_of_year'] = stn_timeseries_df.index.dayofyear
        
        #     # Calculate average value by day of year
        #     average_by_day = stn_timeseries_df.groupby('day_of_year')['ET_corr'].mean()
        
        #     # sum up to an annual total
        #     df1_s = average_by_day.sum()

        #     m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Initial EB Slope'] = 1
        #     m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Corrected EB Slope'] = 1

        # USING REPORTED VALUES FROM DeMeo et al. (2008)
        elif site_id == 'MR':
            df1_s = 1097
            
            m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Initial EB Slope'] = 1
            m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Corrected EB Slope'] = 1
        elif site_id == 'VR':
            df1_s = 1189

            m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Initial EB Slope'] = 1
            m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Corrected EB Slope'] = 1

        # all other sites using full annual timeseries for the site-year ETa
        else:
            
            # filter timeseries dataframe to the current station's water year
            df1 = stn_timeseries_df.loc[start_dt_str: end_dt_str]

            df1_e = df1[['energy', 'flux', 'flux_corr']].dropna(how='any')
            
            coeffs_ini, stats_ini = poly.polyfit(df1_e['energy'], df1_e['flux'], 1, full=True)
            coeffs_cor, stats_cor = poly.polyfit(df1_e['energy'], df1_e['flux_corr'], 1, full=True)
            m_ini = coeffs_ini[1].round(3)
            m_cor = coeffs_cor[1].round(3)
            
            # get annual sum of ETa
            df1_s = df1['ET_corr'].sum().round(2)
        
            # populate the output dataframe with the stats
            if station_typ == 'BR':
                m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Initial EB Slope'] = 1
                m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Corrected EB Slope'] = 1
            elif station_typ == 'EC':
                m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Initial EB Slope'] = m_ini
                m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'Corrected EB Slope'] = m_cor

        # final ETa value filled in the summary table
        m_list_df.loc[(m_list_df['Site'] == site_id) & (m_list_df['Year'] == year), 'ETa (mm/yr)'] = df1_s
        
    except Exception as e:
        print(e)

# export the populated summary table to a CSV
m_list_df.to_csv(os.path.join(table_path, 'flux_data_annual_et_summary.csv'), index=False)
print('exported annual flux data summary table')

exported annual flux data summary table


## 2. Vegetation Index processing
> Google Earth Engine zonal statistic summaries of Landsat Collection 2 Surface Reflectance-derived vegetation indices for flux station footprints/source areas
---

### 2a. Develop Windrose diagram from the flux station wind speed/direction data to help determine which way to shift flux footprint/source area polygons

#### Uses the [Rosely](https://rosely.readthedocs.io/en/latest/install.html) python package

#### Uses growing season months and daytime hours (stations with hourly data only) to determine average wind direction and velocity

In [ ]:
#########################################################################

# eddy covariance datasets (EC) or bowen ratio datasets (BR), files are stored in separate sub-folders
station_data_type = 'EC'

#########################################################################

In [ ]:

table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')

if station_data_type == 'EC':
    in_path = os.path.join(table_path, 'eddy_covariance_station_data')
elif station_data_type == 'BR':
    in_path.os.path.join(table_path, 'bowen_ratio_station_data')


# OpenET Reference ET station Master List
try:
    site_info_df = pd.read_csv(os.path.join(table_path, 'site_year_list.csv'))
except Exception as e:
    print(e)

# list of sites without wind speed/direction data
skip_list = ['SPV_3', 'SNV_1', 'SPV_1', 'SPV_2', 'WRV_1', 'WRV_2', 'SV_4', 'SV_5', 'SV_6', 'ET_1', 'ET_8', 'MR', 'VR', 
             'LMVW', 'UMVW', 'MOVAL', 'UOVLO', 'UOVUP', 'UOVMD', 'GRE', 'SAL', 'GRE_2', 'SAL_2', 'B_11', 'B_01', 'TAM']

# can use a keep list to only process specific stations
# keep_list = ['LONS']

for row_tuple in site_info_df.itertuples():

    site = row_tuple[1].replace('-', '_')

    if keep_list:
        if site not in keep_list:
            continue

    if site in skip_list:
        continue

    yr = row_tuple[3]

    # output file path for the windrose diagram
    out_file_path_pre = os.path.join(in_path, 'windrose')
    out_file_path = os.path.join(out_file_path_pre, f'windrose_{site}_{yr}.html')


    if (site == 'DVDV' or site == 'DVSV'):
        wsc = 'Wind Speed, m/s'
        wdc = 'Wind Direction, compass degrees'
    elif (site == 'AWSR_S' or site == 'AWSR_W'):
        wsc = 'WS (m/s)'
        wdc = 'Wind Direction (degrees)'
    else:
        wsc = 'Wind speed, m/s'
        wdc = 'Wind direction, compass degrees'

    if (site == 'ERG_2' or site == 'LONS'):
        try:
            df = pd.read_csv(os.path.join(in_path, f'raw_daily_{site}.csv'), index_col=0, parse_dates=True, na_values='NO_DATA') 
        except Exception as e:
            print(e)
    else:
        try:
            df = pd.read_csv(os.path.join(in_path, f'raw_subhour_{site}.csv'), index_col=0, parse_dates=True, na_values='NO_DATA')
        except Exception as e:
            print(e)

    # rename data columns
    df = df.rename(columns={wsc: 'ws', wdc: 'wd'})

    # filter to the year for the site-year combo
    df = df[['ws', 'wd']].loc[df.index.year.isin([yr])]

    if not (site == 'ERG_2' or site == 'LONS'):
        # filter to only daytime values
        df = df[['ws', 'wd']].between_time('8:00', '16:00')
    
    # further filter to only growing-season months
    df = df[['ws','wd']].loc[df.index.month.isin([4,5,6,7,8,9,10])]

    # drop nan rows
    df = df[['ws', 'wd']].dropna()
    
    # print(df[['ws', 'wd']].head())
    # print(df[['ws', 'wd']].describe())
    
    # create windrose object
    WR = WindRose(df)
    
    # calculate the stats for the plot
    WR.calc_stats()
    # WR.calc_stats(normed=False, bins=6)
    
    # show the plot
    # WR.plot(output_type='show')
    WR.plot(
        output_type='save', 
        out_file=out_file_path,
        colors='Greens',
        template='plotly_dark',
        colors_reversed=False,
        width=600,
        height=600,
        title=f'Site {site}, {yr} April-October for daytime hours (8:00 - 16:00)'
    )


### 2b. Export spatially-averaged vegetation index values for buffers (i.e., source areas/flux footprints) around flux stations

> Comprehensive list of indices: https://www.indexdatabase.de/db/i.php

**NOTE**: The output from this code block will be sent to your Google Drive. Retrieve the output from your Google Drive, download it, and then store it within the "vegetation_indices" sub-folder for further processing

In [ ]:

table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')

in_path_br_pre = os.path.join(table_path, 'bowen_ratio_station_data')
in_path_br = os.path.join(in_path_br_pre, 'output')

try:
    m_list_df = pd.read_csv(os.path.join(table_path, 'site_year_list.csv'))
except Exception as e:
    print(e)

# keep_list = ['AFD']

# all flags for cloud masking are used
cirrus_flag = True
dilate_flag = True
shadow_flag = True
snow_flag   = True

# buffer size to use around the eddy covariance site (radius in units of meters)
buffer_size = 80

# flag to use custom source areas
custom_flag = True

# flag for custom set following primary wind direction
wind_dir = True

# reduction statistic type (mean vs. median)
reducer_v = 'mean'

# custom source areas for the stations 
if wind_dir:
    buffer_col = ee.FeatureCollection(f'users/bminor/NWI/shapefiles/station_buffer_{buffer_size}m_custom_wind_dir')
else:
    buffer_col = ee.FeatureCollection(f'users/bminor/NWI/shapefiles/station_buffer_{buffer_size}m_custom')

# weighted or unweighted reducer
reduc_type = 'weighted'

# extract VI values using a single path/row or all paths/rows (single or multi)
tiles_pr = 'single'

# day of year start and end for defining which images to process for median VI composite images
doy_st = 152
doy_en = 258


# UTM zone for extractions
utm_z = '11'

# Applies scale factors to Landsat Collection 2 Tier 1 image collection
def applyScaleFactors(img):
    """unscale all bands in a image from the Landsat Level 2 Collection 2 Tier 1 image collection

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    opticalBands = img.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = img.select('ST_B.*').multiply(0.00341802).add(149.0)
    return (
        img
            .addBands(opticalBands, None, True)
            .addBands(thermalBands, None, True)
    )

# Cloud Mask SR using bits
def cloudMask(img):
    """mask out clouds in an image from the Landsat Level 2 Collection 2 Tier 1 image collection using QA bits

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    qa_img = img.select(['QA_PIXEL'])
    cloud_mask = qa_img.rightShift(3).bitwiseAnd(1).neq(0)
    #     .And(qa_img.rightShift(6).bitwiseAnd(3).gte(cloud_confidence)) issue with bit 6
    if cirrus_flag:
        cloud_mask = cloud_mask.Or(qa_img.rightShift(2).bitwiseAnd(1).neq(0))
    if dilate_flag:
        cloud_mask = cloud_mask.Or(qa_img.rightShift(1).bitwiseAnd(1).neq(0))
    if shadow_flag:
        cloud_mask = cloud_mask.Or(qa_img.rightShift(4).bitwiseAnd(1).neq(0))
    if snow_flag:
        cloud_mask = cloud_mask.Or(qa_img.rightShift(5).bitwiseAnd(1).neq(0))

    # Flip to set cloudy pixels to 0 and clear to 1
    return img.updateMask(cloud_mask.Not())


def dateProperties(img):
    """set a new year property on an image

    Args:
        img (ee.Image): a single earth engine image
    
    Returns:
        ee.Image
    """
    return (
        img
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system:index', 'system:time_start', 'system:time_end'])
    )

# function to calculate NDVI 
def srNDVI(img):
    """calculate NDVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red','nir'])
            .expression('((b(1) - b(0)) / (b(1) + b(0)))')
            .rename(['ndvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate EVI
def srEVI(img):
    """calculate EVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(["blue", "red", "nir"])
            .expression('2.5 * (b(2) - b(1)) / (b(2) + (6.0 * b(1)) - (7.5 * b(0)) + 1.0)')
            .rename(['evi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate EVI - 2 (https://www.mdpi.com/2072-4292/16/12/2122)
def srEVI2(img):
    """calculate EVI2 for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(["red", "nir"])
            .expression('2.5 * (b(1) - b(0)) / (b(1) + (2.4 * b(0)) + 1.0)')
            .rename(['evi2'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )
        

# function to calculate empirical MSAVI
def srMSAVI(img):
    """calculate MSAVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    L = (
        img
            .select(["red", "nir"])
            .expression('1 - (2 * 1.06 * (b(1) -b(0)) * (b(1) - 1.06 * b(0))) / (b(1) +b(0))')
    )
    return (
        ee.Image(img
            .select(["red", "nir"])
            .expression('(((b(1) - b(0)) * (1 + L)) / (b(1) + b(0) + L))', {'L': L})
            .rename(['msavi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate inductive MSAVI2
def srMSAVI2(img):
    """calculate MSAVI2 for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(["red", "nir"])
            .expression('(2 * b(1) + 1 - sqrt(pow((2 * b(1) +1),2) - 8 * (b(1) - b(0)))) / 2')
            .rename(['msavi2'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate ATSAVI (https://www.indexdatabase.de/db/i-single.php?id=209)
def srATSAVI(img):
    """calculate ATSAVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red','nir'])
            .expression('1.22 * ((b(1) - (1.22 * b(0)) - 0.03) / ((1.22 * b(1)) + b(0) - (1.22 * 0.03) + (0.08 * (1 + (1.22**2)))))')
            .rename(['atsavi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate CVI chlorophyll vegetation index (https://www.indexdatabase.de/db/i-single.php?id=391)
def srCVI(img):
    """calculate CVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red','nir','green'])
            .expression('(b(1) * (b(0) / (b(2)**2)))')
            .rename(['cvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate CTVI corrected transformed vegetation index (https://www.indexdatabase.de/db/i-single.php?id=244; https://www.researchgate.net/publication/346483699_Hyperspectral_Vegetation_Indices_Calculated_by_Qgis_Using_Landsat_Tm_Image_a_Case_Study_of_Northern_Iceland) 
def srCTVI(img):
    """calculate CTVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red','nir'])
            .expression('((((b(1) - b(0)) / (b(1) + b(0))) + 0.5) / abs(((b(1) - b(0)) / (b(1) + b(0))) + 0.5)) * sqrt(abs(((b(1) - b(0)) / (b(1) + b(0))) + 0.5))')
            .rename(['ctvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate ARVI (https://www.sciencedirect.com/science/article/pii/S2590197420300148#:~:text=To%20correct%20the%20radiance%20in,(%20NIR%20%2B%20RB%20)%20Eq.) 
def srARVI(img):
    """calculate ARVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red','nir','blue'])
            .expression('(b(1) - (b(0) - 1 * (b(2) - b(0)))) / (b(1) + (b(0) - 1 * (b(2) - b(0))))')
            .rename(['arvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate NDVI/Ts (NDVI normalized by Ts)
def srTsNDVI(img):
    """calculate NDVI/surface temperature (Ts) for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red','nir','ts'])
            .expression('(((b(1) - b(0)) / (b(1) + b(0))) / b(2))')
            .rename(['tsndvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate Advanced Vegetation Index (AVI, https://support.climateengine.org/article/165-advanced-vegetation-index-avi)
def srAVI(img):
    """calculate AVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red', 'nir'])
            .expression('(b(1) * (1 - b(0)) * (b(1) - b(0)))**(1/3)')
            .rename(['avi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate Normalized Vegetation Index (NVI, https://farmonaut.com/remote-sensing/mastering-vegetation-indices-ndvi-nvi-and-vari-essential-tools-for-precision-agriculture-and-remote-sensing/)
def srNVI(img):
    """calculate NVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red', 'nir'])
            .expression('b(1) / (b(1) + b(0))')
            .rename(['nvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate Visible Atmospherically Resistant Vegetation Index (VARI, https://farmonaut.com/remote-sensing/top-3-vegetation-indices-for-precision-agriculture)
def srVARI(img):
    """calculate VARI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['green', 'red', 'blue'])
            .expression('(b(0) - b(1)) / (b(0) + b(1) - b(2))')
            .rename(['vari'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

# function to calculate Renormalized Difference Vegetation Index (RDVI, https://doi.org/10.1016/0034-4257(94)00114-3)
def srRDVI(img):
    """calculate RDVI for an image

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    return (
        ee.Image(img
            .select(['red', 'nir'])
            .expression('(b(1) - b(0)) / sqrt(b(1) + b(0))')
            .rename(['rdvi'])
            .set('year', ee.Number(ee.Date(img.get('system:time_start')).get('year')))
            .copyProperties(img, ['system_index', 'system:time_start']))
    )

    
# USGS SR OLI 2 ETM conversions (red, nir, blue, QA_PIXEL)
def sr_oli_2_etm_func(img):
    """applies regression coefficients to adjust Landsat red/nir bandwidths for OLI to match ETM

    Args:
        img (ee.Image): a single earth engine image

    Returns:
        ee.Image
    """
    m = ee.Image.constant([0.9911, 0.9892, 1.0, 1.0, 1.0, 1.0])
    b = ee.Image.constant([0.0099, 0.0070, 1.0, 1.0, 1.0, 1.0])
    return (
        img
            .multiply(m)
            .add(b)
            .copyProperties(img, ['system:index', 'system:time_start'])
    )

    
ftrCollFinal = ee.FeatureCollection([])

# loop through sites and skip any that aren't in keep_list variable
for row_tuple in m_list_df.itertuples():
    
    site_id = row_tuple[1]
    

    year = row_tuple[3]

    # print(f'SITE ID: {site_id}  YEAR: {year} ')

    # custom, wind direction shifted source areas for some site-years
    if (site_id == 'DVSV' and year == 2010):
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .filter(ee.Filter.eq('Year', 2010))
                .first())
        )
    elif (site_id == 'DVSV' and year == 2011):
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .filter(ee.Filter.eq('Year', 2011))
                .first())
        )
    elif (site_id == 'KV-2' and year == 2011):
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .filter(ee.Filter.eq('Year', 2011))
                .first())
        )
    elif (site_id == 'KV-2' and year == 2012):
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .filter(ee.Filter.eq('Year', 2012))
                .first())
        )
    elif (site_id == 'KV-3' and year == 2011):
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .filter(ee.Filter.eq('Year', 2011))
                .first())
        )
    elif (site_id == 'KV-3' and year == 2012):
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .filter(ee.Filter.eq('Year', 2012))
                .first())
        )
    else:
        src_area = (
            ee.Feature(buffer_col
                .filter(ee.Filter.eq('Site', site_id))
                .first())
        )
    
    # phase 2 organization
    lat = row_tuple[5]
    lon = row_tuple[6]
    point = ee.Geometry.Point(lon, lat)
    
    # prepare list of paths and rows for filtering Landsat collection to site
    wrs2_tile = row_tuple[4]
    split = wrs2_tile.split(',')
    
    # list comprehensions to build lists of paths and rows
    path_list_str = [w[2:4] for w in split]
    path_list = [int(i) for i in path_list_str]
    
    row_list_str  = [w[6:8] for w in split]
    row_list = [int(i) for i in row_list_str]

    if tiles_pr == 'single':
        path_list = [path_list[0]]
        row_list = [row_list[0]]

    # need to compute doy averages for certain sites (DeMeo et al., 2008)
    if site_id == 'MR':
        YEAR_RANGE = ee.Filter.calendarRange(2004, 2006, 'year')
    elif site_id == 'VR':
        YEAR_RANGE = ee.Filter.calendarRange(2003, 2004, 'year')
    elif site_id == 'UMVW':
        YEAR_RANGE = ee.Filter.calendarRange(2005, 2006, 'year')
    else:    
        YEAR_RANGE = ee.Filter.calendarRange(year, year, 'year')
        
    #select day of year range to compute Landsat median for each year (June 1 - Sept 15 default)
    DOY_RANGE = ee.Filter.calendarRange(doy_st, doy_en, 'day_of_year')
    
    landsat5 = (
        ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
            .filter(ee.Filter.inList('WRS_PATH',path_list))
            .filter(ee.Filter.inList('WRS_ROW',row_list))
            .filter(YEAR_RANGE)
            .filter(DOY_RANGE)
            .map(applyScaleFactors)
            .select(['SR_B3','SR_B4','SR_B1','SR_B2','ST_B6','QA_PIXEL'],['red','nir','blue','green','ts','QA_PIXEL'])
            .filter(ee.Filter.lt('CLOUD_COVER',70))
    )

    landsat7 = (
        ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
            .filter(ee.Filter.inList('WRS_PATH',path_list))
            .filter(ee.Filter.inList('WRS_ROW',row_list))
            .filter(YEAR_RANGE)
            .filter(DOY_RANGE)
            .map(applyScaleFactors)
            .select(['SR_B3','SR_B4','SR_B1','SR_B2','ST_B6','QA_PIXEL'],['red','nir','blue','green','ts','QA_PIXEL'])
            .filter(ee.Filter.lt('CLOUD_COVER',70))
    )
    
    landsat8 = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .filter(ee.Filter.inList('WRS_PATH',path_list))
            .filter(ee.Filter.inList('WRS_ROW',row_list))
            .filter(YEAR_RANGE)
            .filter(DOY_RANGE)
            .map(applyScaleFactors)
            .select(['SR_B4','SR_B5','SR_B2','SR_B3','ST_B10','QA_PIXEL'],['red','nir','blue','green','ts','QA_PIXEL'])
            .filter(ee.Filter.lt('CLOUD_COVER',70))
            .map(sr_oli_2_etm_func)
    )
    
    
    # merge L5, L7, and L8 collections
    landsat_coll = ee.ImageCollection([])
    landsat_coll = ee.ImageCollection(landsat_coll.merge(landsat5.merge(landsat7.merge(landsat8))))
#     print(landsat_coll.size().getInfo())

    # cloud masking, ndvi calc, set date properties mapping over the merged collection    
    ndvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srNDVI)
            .map(dateProperties)
    )

    evi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srEVI)
            .map(dateProperties)
    )

    evi2_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srEVI2)
            .map(dateProperties)
    )

    msavi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srMSAVI)
            .map(dateProperties)
    )

    msavi2_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srMSAVI2)
            .map(dateProperties)
    )

    atsavi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srATSAVI)
            .map(dateProperties)
    )
        
    cvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srCVI)
            .map(dateProperties)
    )

    ctvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srCTVI)
            .map(dateProperties)
    )

    arvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srARVI)
            .map(dateProperties)
    )

    tsndvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srTsNDVI)
            .map(dateProperties)
    )

    avi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srAVI)
            .map(dateProperties)
    )

    nvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srNVI)
            .map(dateProperties)
    )

    vari_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srVARI)
            .map(dateProperties)
    )

    rdvi_coll = (
        landsat_coll
            .map(cloudMask)
            .map(srRDVI)
            .map(dateProperties)
    )
    
    # reduce collections to median composites
    med_ndvi_comp = ndvi_coll.median()
    med_evi_comp = evi_coll.median()
    med_evi2_comp = evi2_coll.median()
    med_msavi_comp = msavi_coll.median()
    med_msavi2_comp = msavi2_coll.median()
    med_atsavi_comp = atsavi_coll.median()
    med_cvi_comp = cvi_coll.median()
    med_ctvi_comp = ctvi_coll.median()
    med_arvi_comp = arvi_coll.median()
    med_tsndvi_comp = tsndvi_coll.median()
    med_avi_comp = avi_coll.median()
    med_nvi_comp = nvi_coll.median()
    med_vari_comp = vari_coll.median()
    med_rdvi_comp = rdvi_coll.median()

    # prepare gridMET ETo collection to merge with NDVI composite
    if (site_id == 'AFD' or site_id == 'AFS'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year-1)}-11-15', f'{str(year)}-11-15'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year-1)}-11-15', f'{str(year)}-11-15'))
                .select(['prcp'])
        )
    elif site_id == 'AWSR_S':
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-01-04', f'{str(year+1)}-01-04'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-01-04', f'{str(year+1)}-01-04'))
                .select(['prcp'])
        )
    elif site_id == 'AWSR_W':
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-01-01', f'{str(year+1)}-01-01'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-01-01', f'{str(year+1)}-01-01'))
                .select(['prcp'])
        )
    elif site_id == 'B-01':
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-04-01', f'{str(year+1)}-04-01'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-04-01', f'{str(year+1)}-04-01'))
                .select(['prcp'])
        )
    elif (site_id == 'GRE' or site_id == 'SAL' or site_id == 'B-11'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-03-01', f'{str(year+1)}-03-01'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-03-01', f'{str(year+1)}-03-01'))
                .select(['prcp'])
        )
    elif (site_id == 'RAB'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-03-08', f'{str(year+1)}-03-08'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-03-08', f'{str(year+1)}-03-08'))
                .select(['prcp'])
        )
    elif (site_id == 'ERG-2'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-04-24', f'{str(year+1)}-04-24'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-04-24', f'{str(year+1)}-04-24'))
                .select(['prcp'])
        )
    elif (site_id == 'LONS'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year)}-02-13', f'{str(year+1)}-02-13'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year)}-02-13', f'{str(year+1)}-02-13'))
                .select(['prcp'])
        )
    elif (site_id == 'CC'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year-1)}-11-25', f'{str(year)}-11-25'))
                .select(['eto', 'pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year-1)}-11-25', f'{str(year)}-11-25'))
                .select(['prcp'])
        )
    elif site_id == 'MR':

        # using station ET data file to get list of dates for averaging
        try:
            et_df = pd.read_csv(os.path.join(in_path_br, 'MR_daily_data.csv'), index_col='date', parse_dates=True)
        except Exception as e:
            print(e)
            
        et_df = et_df.loc[et_df['ET_gap'] == False]
        datetime_list = et_df.index.date.tolist()
        dates_list = [date_obj.strftime('%Y%m%d') for date_obj in datetime_list]
        
        gridmet_coll_pre = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.inList('system:index', dates_list))
                .select(['eto', 'pr'])
        )

        daymet_coll_pre = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.inList('system:index', dates_list))
                .select(['prcp'])
        )

        days_eto = ee.List.sequence(1, 365)

        def doyAvgEToG(d):
            return (
                gridmet_coll_pre
                    .filter(ee.Filter.calendarRange(d, d, 'day_of_year'))
                    .select(['eto', 'pr'])
                    .mean()
            )

        def doyAvgEToD(d):
            return (
                daymet_coll_pre
                    .filter(ee.Filter.calendarRange(d, d, 'day_of_year'))
                    .select(['prcp'])
                    .mean()
            )

        gridmet_coll = ee.ImageCollection.fromImages(days_eto.map(doyAvgEToG))
        daymet_coll = ee.ImageCollection.fromImages(days_eto.map(doyAvgEToD))
    elif site_id == 'VR':

        # using station ET data file to get list of dates for averaging
        try:
            et_df = pd.read_csv(os.path.join(in_path_br, 'VR_daily_data.csv'), index_col='date', parse_dates=True)
        except Exception as e:
            print(e)
            
        et_df = et_df.loc[et_df['ET_gap'] == False]
        datetime_list = et_df.index.date.tolist()
        dates_list = [date_obj.strftime('%Y%m%d') for date_obj in datetime_list]
        
        gridmet_coll_pre = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.inList('system:index', dates_list))
                .select(['eto', 'pr'])
        )

        daymet_coll_pre = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.inList('system:index', dates_list))
                .select(['prcp'])
        )
        
        days_eto = ee.List.sequence(1, 365)

        def doyAvgEToG(d):
            return (
                gridmet_coll_pre
                    .filter(ee.Filter.calendarRange(d, d, 'day_of_year'))
                    .select(['eto','pr'])
                    .mean()
            )

        def doyAvgEToD(d):
            return (
                daymet_coll_pre
                    .filter(ee.Filter.calendarRange(d, d, 'day_of_year'))
                    .select(['prcp'])
                    .mean()
            )
        
        gridmet_coll = ee.ImageCollection.fromImages(days_eto.map(doyAvgEToG))  
        daymet_coll = ee.ImageCollection.fromImages(days_eto.map(doyAvgEToD))  
    elif (site_id == 'WRV-1' or site_id == 'WRV-2' or site_id == 'UOVMD' or site_id == 'UOVUP' or site_id == 'SNV-1' or site_id == 'SPV-1' or site_id == 'SPV-2' or site_id == 'SPV-3'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year-1)}-09-01', f'{str(year)}-09-01'))
                .select(['eto','pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year-1)}-09-01', f'{str(year)}-09-01'))
                .select(['prcp'])
        )
    elif (site_id == 'EGBRX'):
        gridmet_coll = (
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year-1)}-08-01', f'{str(year)}-08-01'))
                .select(['eto','pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year-1)}-08-01', f'{str(year)}-08-01'))
                .select(['prcp'])
        )        
    else:
        gridmet_coll =(
            ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
                .filter(ee.Filter.date(f'{str(year-1)}-10-01', f'{str(year)}-10-01'))
                .select(['eto','pr'])
        )

        daymet_coll = (
            ee.ImageCollection("NASA/ORNL/DAYMET_V4")
                .filter(ee.Filter.date(f'{str(year-1)}-10-01', f'{str(year)}-10-01'))
                .select(['prcp'])
        )
    
    # reduce collection to water-year total composite
    gridmet_an = gridmet_coll.sum().rename(['eto_an','pr_an'])

    daymet_an = daymet_coll.sum().rename(['prcp_an'])
    
    # merge the two bands for the reducer below
    all_bnds = (
        ee.ImageCollection(med_ndvi_comp.addBands(med_evi_comp).addBands(med_evi2_comp).addBands(med_msavi_comp)
            .addBands(med_msavi2_comp).addBands(med_atsavi_comp).addBands(med_cvi_comp).addBands(med_ctvi_comp)
            .addBands(med_arvi_comp).addBands(med_tsndvi_comp).addBands(med_avi_comp).addBands(med_nvi_comp)
            .addBands(med_vari_comp).addBands(med_rdvi_comp).addBands(gridmet_an).addBands(daymet_an))
    )

    # gridmet_bnds = ee.ImageCollection(gridmet_an)
    # daymet_bnds = ee.ImageCollection(daymet_an)

    # compute mean or median vi value for site buffer from median ndvi composite
    if custom_flag:
        if reduc_type == 'weighted':
            if reducer_v == 'mean':
                def mVIReduc(img):
                    stats_info = (
                        img
                            .reduceRegion(
                                reducer=(
                                    ee.Reducer.mean()
                                        .combine(ee.Reducer.count(), '', True)
                                ),
                                geometry=src_area.geometry(),
                                crs=f'EPSG:326{utm_z}',
                                crsTransform=[30, 0, 15, 0, -30, 15])
                    )
                    return ee.Feature(None, stats_info)
            
            else:
                def mVIReduc(img):
                    stats_info = (
                        img
                            .reduceRegion(
                                reducer=(
                                    ee.Reducer.median()
                                        .combine(ee.Reducer.count(), '', True)
                                ),
                                geometry=src_area.geometry(),
                                crs=f'EPSG:326{utm_z}',
                                crsTransform=[30, 0, 15, 0, -30, 15])
                    )
                    return ee.Feature(None, stats_info)                
                    
        elif reduc_type == 'unweighted':
            if reducer_v == 'mean':
                def mVIReduc(img):
                    stats_info = (
                        img
                            .reduceRegion(
                                reducer=(
                                    ee.Reducer.mean().unweighted()
                                        .combine(ee.Reducer.count().unweighted(), '', True)
                                ),
                                geometry=src_area.geometry(),
                                crs=f'EPSG:326{utm_z}',
                                crsTransform=[30, 0, 15, 0, -30, 15])
                    )
                    return ee.Feature(None, stats_info)
            else:
                def mVIReduc(img):
                        stats_info = (
                            img
                                .reduceRegion(
                                    reducer=(
                                        ee.Reducer.median().unweighted()
                                            .combine(ee.Reducer.count().unweighted(), '', True)
                                    ),
                                    geometry=src_area.geometry(),
                                    crs=f'EPSG:326{utm_z}',
                                    crsTransform=[30, 0, 15, 0, -30, 15])
                        )
                        return ee.Feature(None, stats_info)
    else:
        if reduc_type == 'weighted':
            if reducer_v == 'mean':
                def mVIReduc(img):
                    stats_info = (
                        img
                            .reduceRegion(
                                reducer=(
                                    ee.Reducer.mean()
                                        .combine(ee.Reducer.count(), '', True)
                                ),
                                geometry=point.buffer(buffer_size),
                                crs=f'EPSG:326{utm_z}',
                                crsTransform=[30, 0, 15, 0, -30, 15])
                    )
                    return ee.Feature(None, stats_info)
            else:
                def mVIReduc(img):
                        stats_info = (
                            img
                                .reduceRegion(
                                    reducer=(
                                        ee.Reducer.median()
                                            .combine(ee.Reducer.count(), '', True)
                                    ),
                                    geometry=point.buffer(buffer_size),
                                    crs=f'EPSG:326{utm_z}',
                                    crsTransform=[30, 0, 15, 0, -30, 15])
                        )
                        return ee.Feature(None, stats_info)
                
        elif reduc_type == 'unweighted':
            if reducer_v == 'mean':
                
                def mVIReduc(img):
                    stats_info = (
                        img
                            .reduceRegion(
                                reducer=(
                                    ee.Reducer.mean().unweighted()
                                        .combine(ee.Reducer.count().unweighted(), '', True)
                                ),
                                geometry=point.buffer(buffer_size),
                                crs=f'EPSG:326{utm_z}',
                                crsTransform=[30, 0, 15, 0, -30, 15])
                    )
                    return ee.Feature(None, stats_info)
            else:
                def mVIReduc(img):
                    stats_info = (
                        img
                            .reduceRegion(
                                reducer=(
                                    ee.Reducer.median().unweighted()
                                        .combine(ee.Reducer.count().unweighted(), '', True)
                                ),
                                geometry=point.buffer(buffer_size),
                                crs=f'EPSG:326{utm_z}',
                                crsTransform=[30, 0, 15, 0, -30, 15])
                    )
                    return ee.Feature(None, stats_info)

    # map the function over the image collection
    mean_vi = all_bnds.map(mVIReduc)
    
    # functions for formatting data summary columns
    if reducer_v == 'mean':
        def OrgColl(ftr):
            """sets new properties on a feature
        
            Args:
                ftr (ee.Feature): a single earth engine feature
        
            Returns:
                ee.Feature
            """
            return ftr.set({
                'SITE_ID': site_id,
                'YEAR': year,
                'MEAN_NDVI': ftr.get('ndvi_mean'),
                'MEAN_EVI': ftr.get('evi_mean'),
                'MEAN_EVI2': ftr.get('evi2_mean'),
                'MEAN_MSAVI': ftr.get('msavi_mean'),
                'MEAN_MSAVI2': ftr.get('msavi2_mean'),
                'MEAN_ATSAVI': ftr.get('atsavi_mean'),
                'MEAN_CVI': ftr.get('cvi_mean'),
                'MEAN_CTVI': ftr.get('ctvi_mean'),
                'MEAN_ARVI': ftr.get('arvi_mean'),
                'MEAN_TsNDVI': ftr.get('tsndvi_mean'),
                'MEAN_AVI': ftr.get('avi_mean'),
                'MEAN_NVI': ftr.get('nvi_mean'),
                'MEAN_VARI': ftr.get('vari_mean'),
                'MEAN_RDVI': ftr.get('rdvi_mean'),
                'ETO_AN_GRIDMET': ee.Number(ftr.get('eto_an_mean')).round(),
                'PPT_AN_GRIDMET': ee.Number(ftr.get('pr_an_mean')).round(),
                'PPT_AN_DAYMET': ee.Number(ftr.get('prcp_an_mean')).round(),
                'PIXEL_COUNT': ftr.get('ndvi_count')
            })
    else:
        def OrgColl(ftr):
            """sets new properties on a feature
        
            Args:
                ftr (ee.Feature): a single earth engine feature
        
            Returns:
                ee.Feature
            """
            return ftr.set({
                'SITE_ID': site_id,
                'YEAR': year,
                'MED_NDVI': ftr.get('ndvi_median'),
                'MED_EVI': ftr.get('evi_median'),
                'MED_EVI2': ftr.get('evi2_median'),
                'MED_MSAVI': ftr.get('msavi_median'),
                'MED_MSAVI2': ftr.get('msavi2_median'),
                'MED_ATSAVI': ftr.get('atsavi_median'),
                'MED_CVI': ftr.get('cvi_median'),
                'MED_CTVI': ftr.get('ctvi_median'),
                'MED_ARVI': ftr.get('arvi_median'),
                'MED_TsNDVI': ftr.get('tsndvi_median'),
                'MED_AVI': ftr.get('avi_median'),
                'MED_NVI': ftr.get('nvi_median'),
                'MED_VARI': ftr.get('vari_median'),
                'MED_RDVI': ftr.get('rdvi_median'),
                'ETO_AN_GRIDMET': ee.Number(ftr.get('eto_an_median')).round(),
                'PPT_AN_GRIDMET': ee.Number(ftr.get('pr_an_median')).round(),
                'PPT_AN_DAYMET': ee.Number(ftr.get('prcp_an_median')).round(),
                'PIXEL_COUNT': ftr.get('ndvi_count')
            })
    
    org_coll = ee.FeatureCollection(mean_vi).map(OrgColl)

    # merge the formatted output to the output feature collection as single row for the site-year
    ftrCollFinal = ftrCollFinal.merge(org_coll)
    
# Export a CSV file to Cloud Storage.
if custom_flag:
    if wind_dir:
        filename = f'{reducer_v}_vi_{buffer_size}m_custom_wind_dir_buffer_{reduc_type}_reduction_{tiles_pr}_pathrow_doy_{doy_st}_{doy_en}'
    else:
        filename = f'{reducer_v}_vi_{buffer_size}m_custom_buffer_{reduc_type}_reduction_{tiles_pr}_pathrow_doy_{doy_st}_{doy_en}'
else:
    filename = f'{reducer_v}_vi_{buffer_size}m_buffer_{reduc_type}_reduction_{tiles_pr}_pathrow_doy_{doy_st}_{doy_en}'

# task for exporting stats table
if reducer_v == 'mean':
    ndvi_reduct = ee.batch.Export.table.toDrive(**{
        'collection': ftrCollFinal,
        'description': f'viNVSitesSpatial{reducer_v}_{buffer_size}m_doy_{doy_st}_{doy_en}',
        'folder': 'NWI_BeamerMinor_VI_Analysis',
        'fileNamePrefix': filename,
        'fileFormat': 'CSV',
        'selectors': ['SITE_ID', 'YEAR', 'MEAN_NDVI', 'MEAN_EVI', 'MEAN_EVI2', 'MEAN_MSAVI', 'MEAN_MSAVI2', 'MEAN_ATSAVI', 'MEAN_CVI', 
                      'MEAN_CTVI', 'MEAN_ARVI', 'MEAN_TsNDVI', 'MEAN_AVI', 'MEAN_NVI', 'MEAN_VARI', 'MEAN_RDVI', 'ETO_AN_GRIDMET', 'PPT_AN_GRIDMET', 
                      'PPT_AN_DAYMET','PIXEL_COUNT']
        })
    ndvi_reduct.start()
else:
    ndvi_reduct = ee.batch.Export.table.toDrive(**{
        'collection': ftrCollFinal,
        'description': f'viNVSitesSpatial{reducer_v}_{buffer_size}_m_doy_{doy_st}_{doy_en}',
        'folder': 'NWI_BeamerMinor_VI_Analysis',
        'fileNamePrefix': filename,
        'fileFormat': 'CSV',
        'selectors': ['SITE_ID', 'YEAR', 'MED_NDVI', 'MED_EVI', 'MED_EVI2', 'MED_MSAVI', 'MED_MSAVI2', 'MED_ATSAVI', 'MED_CVI', 
                      'MED_CTVI', 'MED_ARVI', 'MED_TsNDVI', 'MED_AVI', 'MED_NVI', 'MED_VARI', 'MED_RDVI', 'ETO_AN_GRIDMET', 'PPT_AN_GRIDMET', 
                      'PPT_AN_DAYMET','PIXEL_COUNT']
        })
    ndvi_reduct.start()
print(f'task started for {reducer_v} reducer statistic, {buffer_size}m buffers, {reduc_type} reducer, {tiles_pr} WRS_TILES, and start doy:{doy_st} end doy: {doy_en}')

### 2c. Join vegetation index summary with flux data summary for regression analysis
First, define the vegetation index summary file to use for regression analysis

Options:
1. normalized difference vegetation index ([NDVI](https://www.indexdatabase.de/db/i-single.php?id=58))
2. enhanced vegetation index ([EVI](https://www.indexdatabase.de/db/i-single.php?id=16))
3. enhanced vegetation index-2 ([EVI2](https://www.mdpi.com/2072-4292/16/12/2122))
4. modified soil-adjusted vegetation index ([MSAVI](https://www.sciencedirect.com/science/article/abs/pii/0034425794901341))
5. modified soil-adjusted vegetation index-2 ([MSAVI2](https://www.sciencedirect.com/science/article/abs/pii/0034425794901341))
6. adjusted transformed soil-adjusted vegetation index ([ATSAVI](https://www.indexdatabase.de/db/i-single.php?id=209))
7. chlorophyll vegetation index ([CVI](https://www.indexdatabase.de/db/i-single.php?id=391))
8. corrected transformed vegetation index ([CTVI](https://www.indexdatabase.de/db/i-single.php?id=244))
9. atmospherically resistant vegetation index ([ARVI](https://www.indexdatabase.de/db/i-single.php?id=4))
10. ratio of the normalized difference vegetation index and surface temperature (TsNDVI)
11. advanced vegetation index ([AVI](https://support.climateengine.org/article/165-advanced-vegetation-index-avi))
12. normalized vegetation index ([NVI](https://farmonaut.com/remote-sensing/top-3-vegetation-indices-for-precision-agriculture))
13. visible atmospherically resistant vegetation index ([VARI](https://farmonaut.com/remote-sensing/top-3-vegetation-indices-for-precision-agriculture))
14. renormalized difference vegetation index ([RDVI](https://doi.org/10.1016/0034-4257(94)00114-3))

In [4]:
###################################################################

# vegetatin index summary data file name from the previous code block
veg_index_summary_filename = 'mean_vi_80m_custom_wind_dir_buffer_weighted_reduction_single_pathrow_doy_152_258.csv'

# vegetation index to use for regression
vi_type = 'NDVI'

# earth engine reduction statistic type (mean or median)
reducer_v = 'mean'

###################################################################

In [11]:
table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')

in_path = os.path.join(table_path, 'vegetation_indices')

try:
    # flux dataframe
    flux_df = pd.read_csv(os.path.join(table_path, 'flux_data_annual_et_summary.csv'), index_col='Site')

    # vegetation index dataframe
    veg_df = pd.read_csv(os.path.join(in_path, veg_index_summary_filename), usecols=['SITE_ID', f'{reducer_v.upper()}_{vi_type}', 'ETO_AN_GRIDMET', 'PPT_AN_GRIDMET'])
    
    # rename columns
    veg_df = veg_df.rename(columns={
        'SITE_ID': 'Site',
        f'{reducer_v.upper()}_{vi_type}': vi_type,
        'ETO_AN_GRIDMET': 'ETo (mm/yr)',
        'PPT_AN_GRIDMET': 'PPT (mm/yr)'
    })

    # set the index for joining the two
    veg_df = veg_df.set_index('Site')

    # merge the two dataframes together
    df = pd.concat([flux_df, veg_df], axis=1)

    # calculate normalized annual ET
    df['ET*'] = (df['ETa (mm/yr)'] - df['PPT (mm/yr)']) / (df['ETo (mm/yr)'] - df['PPT (mm/yr)'])

    # export table for regression analysis
    df.to_csv(os.path.join(table_path, f'regression_analysis_table_{vi_type.lower()}.csv'))
    
except Exceptions as e:
    print(e)


## 3. Regression Analysis
---

#### read the summary table into a dataframe

In [ ]:
########################################################################

table_path_main = os.path.join(root_path, 'tables')
table_path = os.path.join(table_path_main, 'regression_analysis')


# start/end day of year that was used for averaging vegetation index values
doy_st = 152
doy_en = 258

# buffer size that was used for calculating average vegetation index values
buffer = 80

# single or multi for the Landsat path/rows used (default single)
tiles_pr = 'single'

# flag to use if source areas were shifted based on wind direction data
wind_dir_flag = True

# specify the vegetation index to use for regression
vi_type = 'NDVI'

# specify whether to remove/include outliers, true to exclude and false to include
filter_outlier_flag = True

# set to True for coloring the data pairs based on initial EBC slope 
scatter_point_EBslope_color = False


########################################################################


df = pd.read_csv(os.path.join(table_path, f'regression_analysis_table_{vi_type.lower()}.csv'))

if filter_outlier_flag:  
    df = df.loc[~(df['Site'].isin(['MOVAL', 'SAL-2']))]
    df = df.loc[~((df['Site'] == 'SV-4') & (df['Year'] == 2009))]    

    
df

#### define x and y data for regression analysis 

In [57]:

x = np.array(df[vi_type])

y = np.array(df['ET*'])

#### least-squares linear regression stats

In [ ]:
### linear least-squares curve fit to NDVI-ET* data using numpy.polynomial.polyomial module
# y = B0 + (B1 * x)

print(f'{vi_type} - ET* Relationship:')

# fit a curve and determine the regression coefficients
coeffs, stats = poly.polyfit(x, y, 1, full=True)
c_rounded = np.around(coeffs, 4)      # round values for print statement and plotting special elements
print('regression coefficients b0 and b1 are %s and %s respectively. ' %(c_rounded[0], c_rounded[1]))


# number of parameters (3)
k = len(coeffs)

# sample size
n = len(y)

# mean and standard deviation of explantory variable
u_x = np.mean(x)
x_stdev = np.std(x)
ssxx = np.sum((x - u_x)**2)


# mean and standard deviation of response variable
u_y = np.mean(y)
y_stdev = np.std(y)



# specify new values of the independent variable (NDVI) that we will use to evaluate the function
x_new = np.linspace(0.0, .9, 100)      # starting near x-intercept
       
# evaluate the function using the coefficients determined above and the new x values (this will give us a point
# estimate of the actual mean of ET* when x = x_new, or a prediction of ET* that results from a made observation
# when x = x_new
ffit = poly.polyval(x_new, coeffs)


# predicted y values at observed x values
yhat = (coeffs[1] * x) + coeffs[0]


#### Summary Stats
# https://onlinecourses.science.psu.edu/stat501/node/255
# http://www.okstate.edu/sas/v8/sashtml/ets/chap30/sect19.htm
# http://www2.stat.duke.edu/~tjl13/s101/slides/unit6lec3H.pdf

# residuals
resid = y - yhat

# regression sum of squares
ssr = np.sum((yhat - u_y)**2)
ssr_round = np.around(ssr, 3)         # round values for print statement
print('SSR is %s' %ssr_round)

# sum of square errors (should be equivalent to stats[0])
sse = np.sum((y - yhat)**2)
sse_round = np.around(sse, 3)         # round values for print statement
print('SSE is %s ' %sse_round)
#print('SSE is %s' %round(float(stats[0]), 3)) 

# total sum of squares
ssto = np.sum((y - u_y)**2)
#ssto = n * np.var(y)                 # alternative method
#ssto = ssr + sse                     # alternative method
ssto_round = np.around(ssto, 3)       # round values for print statement
print('SST is %s ' %ssto_round)

# coefficient of determination (R^2)
#rsq = 1 - (sse/ssto)                 # alternative method
rsq = ssr/ssto
rsq_round = np.around(rsq, 4)         # round values for print statement
rsq_plot  = np.around(rsq, 2)         # round values for plotting special elements
print('R^2 is %s ' %rsq_round)

# adjusted coefficient of determination
arsq = 1 - (((n - 1)/(n - k)) * (1 - rsq))
arsq_round = np.around(arsq, 4)       # round values for print statement
print('Adjusted R^2 is %s' %arsq_round)
            
# mean square error
mse = sse/n
mse_round = np.around(mse, 5)         # round values for print statement
print('MSE is %s' %mse_round)

# mean absolute error
mae = (np.sum(np.absolute(y - yhat)))/n
mae_round = np.around(mae, 4)         # round values for print statement
print('MAE is %s' %mae_round)

# root mean squared error
rmse = np.sqrt(np.mean((y - yhat)**2))
#rmse = np.sqrt(mse)                  # alternative method
rmse_round = np.around(rmse, 4)       # round values for print statement
print('RMSE is %s ' %rmse_round)

# residual standard error
rse = np.sqrt((np.sum((y-yhat)**2)) / (n - k))
rse_round = np.around(rse, 4)         # round values for print statement
print('RSE is %s' %rse_round)

#### side analysis, outlier detection
> Skip this code block if "filter_outlier_flag" variable if True

In [ ]:
# Calculate residuals

# Detect outliers using the residuals
# Method 1: Using standard deviation
std_dev = np.std(resid)
outliers = np.abs(resid) > 2 * std_dev  # Consider points beyond 2 standard deviations as outliers

# Method 2: Using the interquartile range (IQR)
# Q1 = np.percentile(resid, 25)
# Q3 = np.percentile(resid, 75)
# IQR = Q3 - Q1
# outliers = (resid < (Q1 - 1.5 * IQR)) | (resid > (Q3 + 1.5 * IQR))

# Plot the data and regression line
plt.scatter(x, y)
plt.plot(x, yhat, color='red')
plt.scatter(x[outliers], y[outliers], color='orange', label='Outliers')
plt.xlabel(vi_type)
plt.ylabel('ET*')
plt.title(f'ET*-{vi_type} w/ Outliers')
plt.show()
# plt.savefig(os.path.join(table_path, f'bmm_{vi_type.lower()}_ETstar_with_outliers_{buffer}m_buffer.png'), dpi=200)

# Print indices of outliers
print("Outlier indices:", np.where(outliers)[0])

#### plot the residuals

In [ ]:
# Plot residuals
plt.scatter(yhat, resid)
plt.xlabel("Predicted values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.axhline(y=0, color='r', linestyle='--')
plt.show()

#### formatted plot of the regression with or without (default) the outliers

In [69]:
### Plot source area mean NDVI as independent variable (x) and ET* as dependent variable (y)

# define the data source for the plot
source_et = ColumnDataSource(data = dict(
            study   = df['Reference'],
            year    = df['Year'],
            site    = df['Site'],
            VI    = df[vi_type],
            ET_star = df['ET*'],
            init_slo = df['Initial EB Slope']
        )
    )

if scatter_point_EBslope_color:
    if filter_outlier_flag:
        # with removal of outliers (SV-4 2009, MOVAL 1998, MOVAL 1999, SAL-2 2007)
        view_1 = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
                                               21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,
                                               39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,
                                               57,58,59,60]))
    else:
        view_1 = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
                                               21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,
                                               39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,
                                               57,58,59,60,61,62,63,64]))

else:
    if filter_outlier_flag:
        # with removal of outliers (SV-4 2009, MOVAL 1998, MOVAL 1999, SAL-2 2007)
        view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
        view_2  = CDSView(filter=IndexFilter([11,12,13,14,15]))                # Spring Valley
        view_3  = CDSView(filter=IndexFilter([16,17]))                         # Carson Valley
        view_4  = CDSView(filter=IndexFilter([18,19,20,21]))                   # Lower Colorado
        view_5  = CDSView(filter=IndexFilter([22,23,24,25,26]))                # Oasis Valley
        view_6  = CDSView(filter=IndexFilter([27,28,29,30,31,32,33,34,35]))    # Walker River
        view_7  = CDSView(filter=IndexFilter([36,37]))                         # Bridgeport
        view_8  = CDSView(filter=IndexFilter([38,39,40,41]))                   # Dixie Valley
        view_9  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Amargosa Desert
        view_10 = CDSView(filter=IndexFilter([46,47,48,49,50,51]))             # Kobeh Valley
        view_11 = CDSView(filter=IndexFilter([52,53]))                         # Amargosa Wild & Scenic River
        view_12 = CDSView(filter=IndexFilter([54]))                            # Corn Creek
        view_13 = CDSView(filter=IndexFilter([55,56,57,58]))                   # Lower Colorado, springs
        view_14 = CDSView(filter=IndexFilter([59,60]))                         # Death Valley        
    else:
        # get subsets of the data source referenced by index number
        view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
        view_2  = CDSView(filter=IndexFilter([11,12,13,14,15,16]))             # Spring Valley
        view_3  = CDSView(filter=IndexFilter([17,18]))                         # Carson Valley
        view_4  = CDSView(filter=IndexFilter([19,20,21,22]))                   # Lower Colorado
        view_5  = CDSView(filter=IndexFilter([23,24,25,26,27,28,29]))          # Oasis Valley
        view_6  = CDSView(filter=IndexFilter([30,31,32,33,34,35,36,37,38,39])) # Walker River
        view_7  = CDSView(filter=IndexFilter([40,41]))                         # Bridgeport
        view_8  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Dixie Valley
        view_9  = CDSView(filter=IndexFilter([46,47,48,49]))                   # Amargosa Desert
        view_10 = CDSView(filter=IndexFilter([50,51,52,53,54,55]))             # Kobeh Valley
        view_11 = CDSView(filter=IndexFilter([56,57]))                         # Amargosa Wild & Scenic River
        view_12 = CDSView(filter=IndexFilter([58]))                            # Corn Creek
        view_13 = CDSView(filter=IndexFilter([59,60,61,62]))                   # Lower Colorado, springs
        view_14 = CDSView(filter=IndexFilter([63,64]))                         # Death Valley 


# create the figure
plot1 = figure(width=800, height=500, title=f"{vi_type}-ET*", tools="")

# Create a color mapper
# c_map = LinearColorMapper(palette=Sunset9, low=min(df['Initial EB Slope']), high=max(df['Initial EB Slope']))
c_map = linear_cmap("init_slo", palette="Sunset9", low=min(df['Initial EB Slope']), high=max(df['Initial EB Slope']))

### data points
if scatter_point_EBslope_color:
    
    pr = plot1.scatter("VI", "ET_star", size=11, color=c_map, legend_label=f"ET* - {vi_type} Pairs", marker='circle', source=source_et, view=view_1)

    # color_bar = ColorBar(color_mapper=c_map, location=(0, 0))
    color_bar = pr.construct_color_bar()
    plot1.add_layout(color_bar, 'right')

    
    # hover tool attributes
    hover_et = HoverTool(
            tooltips=[
                ("Study", "@study"),
                ("Site", "@site"),
                ("Year", "@year"),
                (vi_type, "@VI{0.0000}"),
                ("ET*", "@ET_star{0.0000}"),
                ("Initial EB Slope", "@init_slo{0.00}")],
            renderers=[pr]
        )      
    # add hover tool to the plot
    plot1.add_tools(hover_et)
    plot1.legend.location = [26, 350]

else:

    # BARCAS (Moreo et al., 2007)
    br = plot1.scatter("VI", "ET_star", size=11, color='black', legend_label="BARCAS (Moreo et al., 2007)", marker='diamond', source=source_et, view=view_1)
    # Spring Valley (Arnone et al., 2008)
    sv = plot1.scatter("VI", "ET_star", size=6,  color='black', legend_label="Spring Valley (Arnone et al., 2009)", marker='square', source=source_et, view=view_2)
    # Carson Valley (Maurer et al., 2005)
    cv = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label="Carson Valley (Maurer et al., 2005)", marker='triangle', source=source_et, view=view_3)
    # Bridgeport (DeMeo et al., 2018)
    bp = plot1.scatter("VI", "ET_star", size=6,  color='black', legend_label="Bridgeport (DeMeo, 2018)", marker='circle', source=source_et, view=view_7)
    # Amargosa Desert (Moreo et al., 2017)
    av = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label ="Amargosa Desert (Moreo et al., 2017)", marker='hex', source=source_et, view=view_9)
    # Lower Colorado (DeMeo et al., 2008)
    lc = plot1.scatter("VI", "ET_star", size=11, color='black', legend_label="Lower Colorado (DeMeo et al., 2008)", marker='diamond', source=source_et, view=view_4,  fill_alpha=0)
    # Oasis Valley (Reiner et al., 2002)
    ov = plot1.scatter("VI", "ET_star", size=6,  color='black', legend_label="Oasis Valley (Reiner et al., 2002)", marker='square', source=source_et, view=view_5,  fill_alpha=0)
    # Walker River (Allander et al., 2009)
    wr = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label="Walker River (Allander et al., 2009)", marker='triangle', source=source_et, view=view_6,  fill_alpha=0)
    # Dixie Valley (Garcia et al., 2015)
    dv = plot1.scatter("VI", "ET_star", size=6,  color='black', legend_label="Dixie Valley (Garcia et al., 2015)", marker='circle', source=source_et, view=view_8,  fill_alpha=0)
    # Kobeh Valley (Berger et al., 2016)
    kv = plot1.scatter("VI", "ET_star", size=7,  color='black', legend_label="Kobeh Valley (Berger et al., 2016)", marker='hex', source=source_et, view=view_10, fill_alpha=0)
    # Amargosa Wild & Scenic River (Pavelko & Damar, 2023)
    aw = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label="Amargosa Wild & Scenic River (Pavelko and Damar, 2023)", marker='inverted_triangle', source=source_et, view=view_11)
    # Corn Creek (Moreo et al., 2014)
    cc = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label="Corn Creek (Moreo et al., 2014)", marker='star', source=source_et, view=view_12)
    # Lower Colorado, springs (TBD)
    sp = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label="Lower Colroado, springs (TBD)", marker='plus', source=source_et, view=view_13)
    # Lower Colorado, springs (TBD)
    dt = plot1.scatter("VI", "ET_star", size=8,  color='black', legend_label="Death Valley, (DeMeo et al., 2003)", marker='star', source=source_et, view=view_14, fill_alpha=0)


    # hover tool attributes
    hover_et = HoverTool(
            tooltips=[
                ("Study", "@study"),
                ("Site", "@site"),
                ("Year", "@year"),
                (vi_type, "@VI{0.0000}"),
                ("ET*", "@ET_star{0.0000}"),
                ("Initial EB Slope", "@init_slo{0.00}")],
            renderers=[br, sv, cv, av, bp, lc, ov, wr, dv, kv, aw, cc, sp, dt]
        )
         
    # add hover tool to the plot
    plot1.add_tools(hover_et)



# best fit curve
plot1.line(x_new, ffit, line_width=1.5, line_color="black", legend_label="Best Fit Curve", muted_color="black", muted_alpha=0.05)


### plot aesthetics
plot1.legend.location = [26, 240]
# title and legend
plot1.title.align = "center"
plot1.title.text_font_size = '16pt'
plot1.outline_line_width = 1
plot1.outline_line_color = "black"
plot1.legend.label_text_font = 'sans-serif'
plot1.legend.label_text_font_style = 'bold'
plot1.legend.label_text_font_size = '6pt'
plot1.legend.background_fill_alpha = 0
plot1.legend.border_line_color = None
plot1.legend[0].spacing = 0
plot1.legend.ncols=2
plot1.legend.click_policy = "hide"

# x axis
plot1.xgrid.grid_line_color = None
plot1.xaxis.axis_label = f"Source Area {vi_type}"
plot1.xaxis.axis_label_text_font = 'sans-serif'
plot1.xaxis.axis_label_text_font_style = 'normal'
plot1.xaxis.axis_label_text_font_size = '12pt'
plot1.xaxis.axis_label_standoff = 15
plot1.xaxis.major_tick_in = 6
plot1.xaxis.major_tick_out = 0
plot1.xaxis.major_tick_line_width = 1
plot1.xaxis.minor_tick_in = 4
plot1.xaxis.minor_tick_out = 0
plot1.x_range.start = 0
plot1.x_range.end = 0.9
plot1.xaxis[0].ticker.desired_num_ticks = 8

# y axis
plot1.ygrid.grid_line_color = None
plot1.yaxis.axis_label = "ET*"
plot1.yaxis.axis_label_text_font = 'sans-serif'
plot1.yaxis.axis_label_text_font_style = 'normal'
plot1.yaxis.axis_label_text_font_size = '12pt'
plot1.yaxis.axis_label_standoff = 20
plot1.yaxis.major_tick_in = 6
plot1.yaxis.major_tick_out = 0
plot1.yaxis.major_tick_line_width = 1
plot1.yaxis.minor_tick_in = 4
plot1.yaxis.minor_tick_out = 0
plot1.y_range.start = 0
plot1.y_range.end = 1.401

if filter_outlier_flag:
    if wind_dir_flag:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered.html")
    else:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered.html")
else:
    if wind_dir_flag:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles.html")
    else:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles.html")


output_file(out_file_path, mode='inline')
show(plot1)

#### compute regression coefficients for confidence/prediction intervals

In [ ]:
### Nonlinear least-squares curve fit (second-order polynomial) to NDVI-ET* data using scipy.optimize module
### Produces same coefficients as the polynomial.polyfit method
# y = B0 + (B1 * x)

def model(x, B1, B0):
    return (B1 * x) + (B0)

pars, pcov = curve_fit(model, x, y)     
pars_round = np.around(pars, 4)         # rounded values for print statement and plotting
print('regression coefficients b0 and b1 are %s and %s, respectively. ' %(pars_round[1], pars_round[0]))

alpha = 0.10                            # 90% confidence level
dof   = max(0, n - k)                   # number of degrees of freedom
tval  = t.ppf(1.0 - alpha / 2.0, dof)   # student-t value for the dof and confidence level

# specify new values of the independent variable (NDVI) that we will use to evaluate the function
x_new = np.linspace(0.00, .9, 100)        # starting near x-intercept

# evaluate the function using the coefficients determined above and the new x values
ffit = model(x_new, pars[0], pars[1])


### uncertainties module to find confidence and prediction bands 
# https://pythonhosted.org/uncertainties/
# https://stackoverflow.com/questions/24633664/confidence-interval-for-exponential-curve-fit/37080916#37080916


# a and b (with uncertainty in parameter estimates associated with them)
# same as beta coefficients B1 and B0, respectively
a, b = unc.correlated_values(pars, pcov)

# specify new values of the independent variable (NDVI) that we will use to evaluate the function 
x_star = np.linspace(0, .9, 100)

# predict ET* values at new x values
py = (a * x_star) + b

# unp nominal values 
nom_py   = unp.nominal_values(py)

# unp standard deviations
py_stdev = unp.std_devs(py)

# 90% confidence and prediction bounds

# Beamer 
ci_l   = nom_py - tval * py_stdev
ci_u   = nom_py + tval * py_stdev
pi_l   = nom_py - tval * np.sqrt(rse**2 + py_stdev**2)
pi_u   = nom_py + tval * np.sqrt(rse**2 + py_stdev**2)


ci_low = nom_py - tval * rse * np.sqrt(    (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))
ci_upp = nom_py + tval * rse * np.sqrt(    (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))
pi_low = nom_py - tval * rse * np.sqrt(1 + (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))
pi_upp = nom_py + tval * rse * np.sqrt(1 + (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))

#print('a and b are %s and %s, respectively' %(a,b))
#print('tvalue is %s' %tval)
#print('standard deviation is %s' %py_stdev)
#print('rse is %s'%rse)

In [ ]:
### Confidence and Prediction interval beta coefficients

# function/model to produce beta coefficients of confidence and prediction intervals
def conf_m(x_star, B1, B0):
    return (B1 * x_star) + (B0)

ci_l_pars, ci_l_pcov = curve_fit(conf_m, x_star, ci_l)     
ci_l_pars_round = np.around(ci_l_pars, 4)                    # rounded values for print statement and plotting
ci_u_pars, ci_u_pcov = curve_fit(conf_m, x_star, ci_u)     
ci_u_pars_round = np.around(ci_u_pars, 4)                    # rounded values for print statement and plotting
pi_l_pars, pi_l_pcov = curve_fit(conf_m, x_star, pi_l)     
pi_l_pars_round = np.around(pi_l_pars, 4)                    # rounded values for print statement and plotting
pi_u_pars, pi_u_pcov = curve_fit(conf_m, x_star, pi_u)     
pi_u_pars_round = np.around(pi_u_pars, 4)                    # rounded values for print statement and plotting
print('CI-Lower beta coefficients b0 and b1 are %s and %s, respectively. ' %(ci_l_pars_round[1], ci_l_pars_round[0]))
print('CI-Upper beta coefficients b0 and b2 are %s and %s, respectively. ' %(ci_u_pars_round[1], ci_u_pars_round[0]))
print('PI-Lower beta coefficients b0 and b2 are %s and %s, respectively. ' %(pi_l_pars_round[1], pi_l_pars_round[0]))
print('PI-Upper beta coefficients b0 and b2 are %s and %s, respectively. ' %(pi_u_pars_round[1], pi_u_pars_round[0]))

In [62]:
### Plot best fit curve, 90% confidence intervals, and 90% prediction intervals


# define the data source for the plot
source_ci = ColumnDataSource(data = dict(
            study   = df['Reference'],
            year    = df['Year'],
            site    = df['Site'],
            VI    = df[vi_type],
            ET_star = df['ET*']
        )
    )

# get subsets of the data source referenced by index number
# view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
# view_2  = CDSView(filter=IndexFilter([11,12,13,14,15,16]))             # Spring Valley
# view_3  = CDSView(filter=IndexFilter([17,18]))                         # Carson Valley
# view_4  = CDSView(filter=IndexFilter([19,20,21,22]))                   # Lower Colorado
# view_5  = CDSView(filter=IndexFilter([23,24,25,26,27,28,29]))          # Oasis Valley
# view_6  = CDSView(filter=IndexFilter([30,31,32,33,34,35,36,37,38,39])) # Walker River
# view_7  = CDSView(filter=IndexFilter([40,41]))                         # Bridgeport
# view_8  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Dixie Valley
# view_9  = CDSView(filter=IndexFilter([46,47,48,49]))                   # Amargosa Desert
# view_10 = CDSView(filter=IndexFilter([50,51,52,53,54,55]))             # Kobeh Valley
# view_11 = CDSView(filter=IndexFilter([56,57]))                         # Amargosa Wild & Scenic River
# view_12 = CDSView(filter=IndexFilter([58]))                            # Corn Creek
# view_13 = CDSView(filter=IndexFilter([59,60,61,62]))                   # Lower Colorado, springs
# view_14 = CDSView(filter=IndexFilter([63,64]))                         # Death Valley 

# with removal of outliers (SV-4 2009, MOVAL 1998, MOVAL 1999, SAL-2 2007)
view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
view_2  = CDSView(filter=IndexFilter([11,12,13,14,15]))                # Spring Valley
view_3  = CDSView(filter=IndexFilter([16,17]))                         # Carson Valley
view_4  = CDSView(filter=IndexFilter([18,19,20,21]))                   # Lower Colorado
view_5  = CDSView(filter=IndexFilter([22,23,24,25,26]))                # Oasis Valley
view_6  = CDSView(filter=IndexFilter([27,28,29,30,31,32,33,34,35]))    # Walker River
view_7  = CDSView(filter=IndexFilter([36,37]))                         # Bridgeport
view_8  = CDSView(filter=IndexFilter([38,39,40,41]))                   # Dixie Valley
view_9  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Amargosa Desert
view_10 = CDSView(filter=IndexFilter([46,47,48,49,50,51]))             # Kobeh Valley
view_11 = CDSView(filter=IndexFilter([52,53]))                         # Amargosa Wild & Scenic River
view_12 = CDSView(filter=IndexFilter([54]))                            # Corn Creek
view_13 = CDSView(filter=IndexFilter([55,56,57,58]))                   # Lower Colorado, springs
view_14 = CDSView(filter=IndexFilter([59,60]))                         # Death Valley 

# create the figure
plot2 = figure(width=800, height=500, title="Confidence and Prediction Intervals", tools="")

# best fit curve
plot2.line(x_new, ffit, line_width=1.5, line_color='#000000', legend_label="Best Fit Curve")


### data points
# BARCAS
br = plot2.scatter("VI", "ET_star", marker='diamond', size=11, color="black", source=source_ci, view=view_1)
# Spring Valley
sv = plot2.scatter("VI", "ET_star", marker='square', size=6,  color="black", source=source_ci, view=view_2)
# Carson Valley
cv = plot2.scatter("VI", "ET_star", marker='triangle', size=8,  color="black", source=source_ci, view=view_3)
# Bridgeport
bp = plot2.scatter("VI", "ET_star",   marker='circle', size=6,  color="black", source=source_ci, view=view_7)
# Amargosa 
av = plot2.scatter("VI", "ET_star",     marker='hex', size=8,  color="black", source=source_ci, view=view_9)
# Lower Colorado
lc = plot2.scatter("VI", "ET_star", marker='diamond', size=11, color="black", source=source_ci, view=view_4,  fill_alpha=0)
# Oasis Valley
ov = plot2.scatter("VI", "ET_star", marker='square', size=6,  color="black", source=source_ci, view=view_5,  fill_alpha=0)
# Walker River
wr = plot2.scatter("VI", "ET_star", marker='triangle', size=8,  color="black", source=source_ci, view=view_6,  fill_alpha=0)
# Dixie Valley
dv = plot2.scatter("VI", "ET_star", marker='circle', size=6,  color="black", source=source_ci, view=view_8,  fill_alpha=0)
# Kobeh Valley 
kv = plot2.scatter("VI", "ET_star",  marker='hex', size=7,  color="black", source=source_ci, view=view_10, fill_alpha=0)
# Amargosa Desert
aw = plot2.scatter("VI", "ET_star",  marker='inverted_triangle', size=8,  color="black", source=source_ci, view=view_11)
# Corn Creek
cc = plot2.scatter("VI", "ET_star",  marker='star', size=8,  color="black", source=source_ci, view=view_12)
# Lower Colorado, springs
sp = plot2.scatter("VI", "ET_star",  marker='plus', size=8,  color="black", source=source_ci, view=view_13)
# Lower Colorado, springs
dt = plot2.scatter("VI", "ET_star",  marker='star', size=8,  color="black", source=source_ci, view=view_14, fill_alpha=0)

# 90% confidence and prediction bands
plot2.line(x_star, pi_l, line_width=1.3, line_color="black", line_dash='dashed', legend_label='90% PI')
plot2.line(x_star, pi_u, line_width=1.3, line_color="black", line_dash='dashed')
plot2.line(x_star, ci_l, line_width=1.3, line_color="black", line_dash='dotted', legend_label='90% CI')
plot2.line(x_star, ci_u, line_width=1.3, line_color="black", line_dash='dotted')

# hover tool attributes
hover_ci = HoverTool(
        tooltips=[
            ("Study", "@study"),
            ("Site", "@site"),
            ("Year", "@year"),
            (vi_type, "@VI{0.0000}"),
            ("ET*", "@ET_star{0.0000}")],
        renderers=[br, sv, cv, bp, av, lc, ov, wr, dv, kv, aw, cc, sp, dt]
    )
    
# add hover tool to the plot
plot2.add_tools(hover_ci)


### plot aesthetics

# title and legend
plot2.title.align = "center"
plot2.title.text_font_size = '16pt'
plot2.outline_line_width = 1
plot2.outline_line_color = "black"
plot2.legend.location = [40,320]
plot2.legend.label_text_font ='sans-serif'
plot2.legend.label_text_font_style = 'bold'
plot2.legend.label_text_font_size = '10pt'
plot2.legend.border_line_color = None
plot2.legend.spacing = 1

# x axis
plot2.xgrid.grid_line_color = None
plot2.xaxis.axis_label = f"Source Area {vi_type}"
plot2.xaxis.axis_label_text_font ='sans-serif'
plot2.xaxis.axis_label_text_font_style = 'normal'
plot2.xaxis.axis_label_text_font_size = '12pt'
plot2.xaxis.axis_label_standoff = 15
plot2.xaxis.major_tick_in = 6
plot2.xaxis.major_tick_out = 0
plot2.xaxis.major_tick_line_width = 1
plot2.xaxis.minor_tick_in = 4
plot2.xaxis.minor_tick_out = 0
plot2.x_range.start = 0
plot2.x_range.end = 0.9
plot2.xaxis[0].ticker.desired_num_ticks = 9

# y axis
plot2.yaxis.axis_label = "ET*"
plot2.yaxis.axis_label_text_font ='sans-serif'
plot2.yaxis.axis_label_text_font_style = 'normal'
plot2.yaxis.axis_label_text_font_size = '12pt'
plot2.yaxis.axis_label_standoff = 20
plot2.ygrid.grid_line_color = None
plot2.yaxis.major_tick_in = 6
plot2.yaxis.major_tick_out = 0
plot2.yaxis.major_tick_line_width = 1
plot2.yaxis.minor_tick_in = 4
plot2.yaxis.minor_tick_out = 0
plot2.y_range.start = 0
plot2.y_range.end = 1.401

if filter_outlier_flag:
    if wind_dir_flag:
        out_file_path_ci = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered_confidence_intervals.html")
    else:
        out_file_path_ci = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered_confidence_intervals.html")
else:
    if wind_dir_flag:
        out_file_path_ci = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_confidence_intervals.html")
    else:
        out_file_path_ci = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_confidence_intervals.html") 

output_file(out_file_path_ci, mode='inline')
show(plot2)

#### physics-constrained formatted plot of the regression (final version)

In [ ]:
### Nonlinear least-squares curve fit (second-order polynomial) to NDVI-ET* data using scipy.optimize module
### Produces same coefficients as the polynomial.polyfit method
# y = B0 + (B1 * x)

print(f'{vi_type} - ET* Relationship:')

def model(x, B1, B0):
    return (B1 * x) + (B0)

pars, pcov = curve_fit(model, x, y, bounds=([0,-.04],[3,-0.035]))   
pars_round = np.around(pars, 4)         # rounded values for print statement and plotting
print('regression coefficients b0 and b1 are %s and %s, respectively. ' %(pars_round[1], pars_round[0]))
alpha = 0.10                            # 90% confidence level
dof   = max(0, n - k)                   # number of degrees of freedom
tval  = t.ppf(1.0 - alpha / 2.0, dof)   # student-t value for the dof and confidence level
print(tval)
print(k)
# specify new values of the independent variable (NDVI) that we will use to evaluate the function
x_new = np.linspace(0.00, .9, 100)        # starting near x-intercept

# evaluate the function using the coefficients determined above and the new x values
ffit = model(x_new, pars[0], pars[1])

yhatt = (pars[0] * x) + pars[1]

# mean and standard deviation of explantory variable
u_x = np.mean(x)
x_stdev = np.std(x)
ssxx = np.sum((x - u_x)**2)


### uncertainties module to find confidence and prediction bands 
# https://pythonhosted.org/uncertainties/
# https://stackoverflow.com/questions/24633664/confidence-interval-for-exponential-curve-fit/37080916#37080916


# a and b (with uncertainty in parameter estimates associated with them)
# same as beta coefficients B1 and B0, respectively
a, b = unc.correlated_values(pars, pcov)

# specify new values of the independent variable (NDVI) that we will use to evaluate the function 
x_star = np.linspace(0, .9, 100)

# predict ET* values at new x values
py = (a * x_star) + b

# unp nominal values 
nom_py   = unp.nominal_values(py)

# unp standard deviations
py_stdev = unp.std_devs(py)


################################
# need R2 of constrained regression


# mean and standard deviation of response variable
u_y = np.mean(y)
y_stdev = np.std(y)


#### Summary Stats
# https://onlinecourses.science.psu.edu/stat501/node/255
# http://www.okstate.edu/sas/v8/sashtml/ets/chap30/sect19.htm
# http://www2.stat.duke.edu/~tjl13/s101/slides/unit6lec3H.pdf

# residuals
resid = y - yhatt

# regression sum of squares
ssr = np.sum((yhatt - u_y)**2)
ssr_round = np.around(ssr, 3)         # round values for print statement
print('SSR is %s' %ssr_round)

# sum of square errors (should be equivalent to stats[0])
sse = np.sum((y - yhatt)**2)
sse_round = np.around(sse, 3)         # round values for print statement
print('SSE is %s ' %sse_round)
#print('SSE is %s' %round(float(stats[0]), 3)) 

# total sum of squares
ssto = np.sum((y - u_y)**2)
#ssto = n * np.var(y)                 # alternative method
#ssto = ssr + sse                     # alternative method
ssto_round = np.around(ssto, 3)       # round values for print statement
print('SST is %s ' %ssto_round)

# coefficient of determination (R^2)
rsq = 1 - (sse/ssto)                 # alternative method
# rsq = ssr/ssto
rsq_round = np.around(rsq, 4)         # round values for print statement
rsq_plot  = np.around(rsq, 2)         # round values for plotting special elements
print('R^2 is %s ' %rsq_round)


# adjusted coefficient of determination
arsq = 1 - (((n - 1)/(n - k)) * (1 - rsq))
arsq_round = np.around(arsq, 4)       # round values for print statement
print('Adjusted R^2 is %s' %arsq_round)
            
# mean square error
mse = sse/n
mse_round = np.around(mse, 5)         # round values for print statement
print('MSE is %s' %mse_round)

# mean absolute error
mae = (np.sum(np.absolute(y - yhatt)))/n
mae_round = np.around(mae, 4)         # round values for print statement
print('MAE is %s' %mae_round)

# root mean squared error
rmse = np.sqrt(np.mean((y - yhatt)**2))
#rmse = np.sqrt(mse)                  # alternative method
rmse_round = np.around(rmse, 4)       # round values for print statement
print('RMSE is %s ' %rmse_round)

# residual standard error
rse = np.sqrt((np.sum((y-yhatt)**2)) / (n - k))
rse_round = np.around(rse, 4)         # round values for print statement
print('RSE is %s' %rse_round)

#################################

# 90% confidence and prediction bounds

# Beamer 
ci_l   = nom_py - tval * py_stdev
ci_u   = nom_py + tval * py_stdev
pi_l   = nom_py - tval * np.sqrt(rse**2 + py_stdev**2)
pi_u   = nom_py + tval * np.sqrt(rse**2 + py_stdev**2)

# some other equations from different source
# https://www.youtube.com/watch?feature=player_embedded&v=qVCQi0KPR0s
# https://library2.lincoln.ac.nz/documents/Analysing-the-Variance.pdf


ci_low = nom_py - tval * rse * np.sqrt(    (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))
ci_upp = nom_py + tval * rse * np.sqrt(    (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))
pi_low = nom_py - tval * rse * np.sqrt(1 + (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))
pi_upp = nom_py + tval * rse * np.sqrt(1 + (1/n) + (((x_star - u_x)**2) / ((n - 1) * (x_stdev**2))))

#print('a and b are %s and %s, respectively' %(a,b))
#print('tvalue is %s' %tval)
#print('standard deviation is %s' %py_stdev)
#print('rse is %s'%rse)

In [72]:
### Plot source area mean NDVI as independent variable (x) and ET* as dependent variable (y)
### Forced Intercept


# define the data source for the plot
source_ci = ColumnDataSource(data = dict(
            study   = df['Reference'],
            year    = df['Year'],
            site    = df['Site'],
            VI    = df[vi_type],
            ET_star = df['ET*']
        )
    )

# get subsets of the data source referenced by index number
# view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
# view_2  = CDSView(filter=IndexFilter([11,12,13,14,15,16]))             # Spring Valley
# view_3  = CDSView(filter=IndexFilter([17,18]))                         # Carson Valley
# view_4  = CDSView(filter=IndexFilter([19,20,21,22]))                   # Lower Colorado
# view_5  = CDSView(filter=IndexFilter([23,24,25,26,27,28,29]))          # Oasis Valley
# view_6  = CDSView(filter=IndexFilter([30,31,32,33,34,35,36,37,38,39])) # Walker River
# view_7  = CDSView(filter=IndexFilter([40,41]))                         # Bridgeport
# view_8  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Dixie Valley
# view_9  = CDSView(filter=IndexFilter([46,47,48,49]))                   # Amargosa Desert
# view_10 = CDSView(filter=IndexFilter([50,51,52,53,54,55]))             # Kobeh Valley
# view_11 = CDSView(filter=IndexFilter([56,57]))                         # Amargosa Wild & Scenic River
# view_12 = CDSView(filter=IndexFilter([58]))                            # Corn Creek
# view_13 = CDSView(filter=IndexFilter([59,60,61,62]))                   # Lower Colorado, springs
# view_14 = CDSView(filter=IndexFilter([63,64]))                         # Death Valley 

# with removal of outliers (SV-4 2009, MOVAL 1998, MOVAL 1999, SAL-2 2007)
view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
view_2  = CDSView(filter=IndexFilter([11,12,13,14,15]))                # Spring Valley
view_3  = CDSView(filter=IndexFilter([16,17]))                         # Carson Valley
view_4  = CDSView(filter=IndexFilter([18,19,20,21]))                   # Lower Colorado
view_5  = CDSView(filter=IndexFilter([22,23,24,25,26]))                # Oasis Valley
view_6  = CDSView(filter=IndexFilter([27,28,29,30,31,32,33,34,35]))    # Walker River
view_7  = CDSView(filter=IndexFilter([36,37]))                         # Bridgeport
view_8  = CDSView(filter=IndexFilter([38,39,40,41]))                   # Dixie Valley
view_9  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Amargosa Desert
view_10 = CDSView(filter=IndexFilter([46,47,48,49,50,51]))             # Kobeh Valley
view_11 = CDSView(filter=IndexFilter([52,53]))                         # Amargosa Wild & Scenic River
view_12 = CDSView(filter=IndexFilter([54]))                            # Corn Creek
view_13 = CDSView(filter=IndexFilter([55,56,57,58]))                   # Lower Colorado, springs
view_14 = CDSView(filter=IndexFilter([59,60]))                         # Death Valley 

# create the figure
plotv = figure(width=800, height=500, title="", tools="")



### data points
# BARCAS (Moreo et al., 2007)
br = plotv.scatter("VI", "ET_star", marker='diamond', size=11, color="black", legend_label="BARCAS (Moreo et al., 2007)",          source=source_ci, view=view_1)
# Spring Valley (Arnone et al., 2008)
sv = plotv.scatter("VI", "ET_star", marker='square', size=6,  color="black", legend_label="Spring Valley (Arnone et al., 2008)",  source=source_ci, view=view_2)
# Carson Valley (Maurer et al., 2005)
cv = plotv.scatter("VI", "ET_star", marker='triangle', size=8,  color="black", legend_label="Carson Valley (Maurer et al., 2005)",  source=source_ci, view=view_3)
# Bridgeport (DeMeo et al., 2018)
bp = plotv.scatter("VI", "ET_star", marker='circle',  size=6,  color="black", legend_label="Bridgeport (DeMeo, 2018)",             source=source_ci, view=view_7)
# Amargosa (Moreo et al., 2017)
av = plotv.scatter("VI", "ET_star", marker='hex',   size=8,  color="black", legend_label = "Amargosa Desert (Moreo et al., 2017)",      source=source_ci, view=view_9)
# Lower Colorado (DeMeo et al., 2008)
lc = plotv.scatter("VI", "ET_star", marker='diamond', size=11, color="black", legend_label="Lower Colorado (DeMeo et al., 2008)",  source=source_ci, view=view_4,  fill_alpha=0)
# Oasis Valley (Reiner et al., 2002)
ov = plotv.scatter("VI", "ET_star", marker='square', size=6,  color="black", legend_label="Oasis Valley (Reiner et al., 2002)",   source=source_ci, view=view_5,  fill_alpha=0)
# Walker River (Allander et al., 2009)
wr = plotv.scatter("VI", "ET_star", marker='triangle', size=8,  color="black", legend_label="Walker River (Allander et al., 2009)", source=source_ci, view=view_6,  fill_alpha=0)
# Dixie Valley (Garcia et al., 2015)
dv = plotv.scatter("VI", "ET_star", marker='circle', size=6,  color="black", legend_label="Dixie Valley (Garcia et al., 2015)",   source=source_ci, view=view_8,  fill_alpha=0)
# Kobeh Valley (Berger et al., 2016)
kv = plotv.scatter("VI", "ET_star", marker='hex', size=7, color="black", legend_label="Kobeh Valley (Berger et al., 2016)",   source=source_ci, view=view_10, fill_alpha=0)
# Amargosa Wild & Scenic River (Pavelko & Damar, 2023)
aw = plotv.scatter("VI", "ET_star", marker='inverted_triangle', size=8,  color="black", legend_label="Amargosa Wild & Scenic River (Pavelko & Damar, 2023)",   source=source_ci, view=view_11)
# Amargosa Wild & Scenic River (Pavelko & Damar, 2023)
cc = plotv.scatter("VI", "ET_star", marker='star', size=8,  color="black", legend_label="Corn Creek (Moreo et al., 2014)",   source=source_ci, view=view_12)
# Amargosa Wild & Scenic River (Pavelko & Damar, 2023)
sp = plotv.scatter("VI", "ET_star", marker='plus', size=8,  color="black", legend_label="Lower Colorado, springs (TBD)",   source=source_ci, view=view_13)
# Death valley (DeMeo et al., 2003)
dt = plotv.scatter("VI", "ET_star", marker='plus', size=8,  color="black", legend_label="Death Valley (DeMeo et al., 2003)",   source=source_ci, view=view_14, fill_alpha=0)

# best fit curve
plotv.line(x_new, ffit, line_width=1.5, line_color="black", legend_label="Best Fit Curve", muted_color="black", muted_alpha=0.05)


# hover tool attributes
hover_ci = HoverTool(
        tooltips=[
            ("Study", "@study"),
            ("Site", "@site"),
            ("Year", "@year"),
            (vi_type, "@VI{0.0000}"),
            ("ET*", "@ET_star{0.0000}")],
        renderers=[br, sv, cv, av, bp, lc, ov, wr, dv, kv, aw, cc, sp, dt]
    )
     
# add hover tool to the plot
plotv.add_tools(hover_ci)

### plot aesthetics

# title and legend
plotv.title.align = "center"
plotv.title.text_font_size = '16pt'
plotv.outline_line_width = 1
plotv.outline_line_color = "black"
plotv.legend.location = [26, 205]
plotv.legend.label_text_font ='sans-serif'
plotv.legend.label_text_font_style ='bold'
plotv.legend.label_text_font_size = '6pt'
plotv.legend.background_fill_alpha = 0
plotv.legend.border_line_color = None
plotv.legend.spacing = 0
plotv.legend.ncols = 2
plotv.legend.click_policy = "hide"

# Equation and R2
label_e = Label(x=0.56,y=0.16,text=f'ET* = -0.035 + {pars_round[0]}({vi_type})',text_color='black',text_font_size='9pt',text_font='sans-serif',text_font_style='normal')
#label_r = Label(x=0.54,y=1.0,text='R2 = 0.84',text_color='black',text_font_size='8pt',text_font='Arial')

plotv.add_layout(label_e)
#plotv.add_layout(label_r)

# x axis
plotv.xgrid.grid_line_color = None
plotv.xaxis.axis_label = f"Source Area {vi_type}"
plotv.xaxis.axis_label_text_font ='sans-serif'
plotv.xaxis.axis_label_text_font_style = 'normal'
plotv.xaxis.axis_label_text_font_size = '12pt'
plotv.xaxis.axis_label_standoff = 15
plotv.xaxis.major_tick_in = 6
plotv.xaxis.major_tick_out = 0
plotv.xaxis.major_tick_line_width = 1
plotv.xaxis.minor_tick_in = 4
plotv.xaxis.minor_tick_out = 0
plotv.x_range.start = 0
plotv.x_range.end = 0.9
plotv.xaxis[0].ticker.desired_num_ticks = 8

# y axis
plotv.ygrid.grid_line_color = None
plotv.yaxis.axis_label = "ET*"
plotv.yaxis.axis_label_text_font = 'sans-serif'
plotv.yaxis.axis_label_text_font_style = 'normal'
plotv.yaxis.axis_label_text_font_size = '12pt'
plotv.yaxis.axis_label_standoff = 20
plotv.yaxis.major_tick_in = 6
plotv.yaxis.major_tick_out = 0
plotv.yaxis.major_tick_line_width = 1
plotv.yaxis.minor_tick_in = 4
plotv.yaxis.minor_tick_out = 0
plotv.y_range.start = 0
plotv.y_range.end = 1.201

plotv.min_border_top = 65
plotv.min_border_bottom = 65
plotv.min_border_left = 65
plotv.min_border_right = 50

plotv.toolbar.logo = None
plotv.toolbar_location = None

if filter_outlier_flag:
    if wind_dir_flag:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered_constrained.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained.png")
    else:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered_constrained.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained.png")
else:
    if wind_dir_flag:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained.png")
    else:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained.png")

output_file(out_file_path, mode='inline')
export_png(plotv, filename=png_file_path)
show(plotv)

In [ ]:
### Confidence and Prediction interval beta coefficients

# function/model to produce beta coefficients of confidence and prediction intervals
def conf_m(x_star, B1, B0):
    return (B1 * x_star) + (B0)


ci_l_pars, ci_l_pcov = curve_fit(conf_m, x_star, ci_l)     
ci_l_pars_round = np.around(ci_l_pars, 4)                    # rounded values for print statement and plotting
ci_u_pars, ci_u_pcov = curve_fit(conf_m, x_star, ci_u)     
ci_u_pars_round = np.around(ci_u_pars, 4)                    # rounded values for print statement and plotting
pi_l_pars, pi_l_pcov = curve_fit(conf_m, x_star, pi_l)     
pi_l_pars_round = np.around(pi_l_pars, 4)                    # rounded values for print statement and plotting
pi_u_pars, pi_u_pcov = curve_fit(conf_m, x_star, pi_u)     
pi_u_pars_round = np.around(pi_u_pars, 4)                    # rounded values for print statement and plotting
print('CI-Lower beta coefficients b0 and b1 are %s and %s, respectively. ' %(ci_l_pars_round[1], ci_l_pars_round[0]))
print('CI-Upper beta coefficients b0 and b2 are %s and %s, respectively. ' %(ci_u_pars_round[1], ci_u_pars_round[0]))
print('PI-Lower beta coefficients b0 and b2 are %s and %s, respectively. ' %(pi_l_pars_round[1], pi_l_pars_round[0]))
print('PI-Upper beta coefficients b0 and b2 are %s and %s, respectively. ' %(pi_u_pars_round[1], pi_u_pars_round[0]))

In [76]:
### Plot best fit curve, 90% confidence intervals, and 90% prediction intervals


# define the data source for the plot
source_ci = ColumnDataSource(data = dict(
            study   = df['Reference'],
            year    = df['Year'],
            site    = df['Site'],
            VI    = df[vi_type],
            ET_star = df['ET*']
        )
    )

# get subsets of the data source referenced by index number
# view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
# view_2  = CDSView(filter=IndexFilter([11,12,13,14,15,16]))             # Spring Valley
# view_3  = CDSView(filter=IndexFilter([17,18]))                         # Carson Valley
# view_4  = CDSView(filter=IndexFilter([19,20,21,22]))                   # Lower Colorado
# view_5  = CDSView(filter=IndexFilter([23,24,25,26,27,28,29]))          # Oasis Valley
# view_6  = CDSView(filter=IndexFilter([30,31,32,33,34,35,36,37,38,39])) # Walker River
# view_7  = CDSView(filter=IndexFilter([40,41]))                         # Bridgeport
# view_8  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Dixie Valley
# view_9  = CDSView(filter=IndexFilter([46,47,48,49]))                   # Amargosa Desert
# view_10 = CDSView(filter=IndexFilter([50,51,52,53,54,55]))             # Kobeh Valley
# view_11 = CDSView(filter=IndexFilter([56,57]))                         # Amargosa Wild & Scenic River
# view_12 = CDSView(filter=IndexFilter([58])])                            # Corn Creek
# view_13 = CDSView(filter=IndexFilter([59,60,61,62]))                   # Lower Colorado, springs
# view_14 = CDSView(filter=IndexFilter([63,64]))                         # Death Valley 

# with removal of outliers (SV-4 2009, MOVAL 1998, MOVAL 1999, SAL-2 2007)
view_1  = CDSView(filter=IndexFilter([0,1,2,3,4,5,6,7,8,9,10]))        # BARCAS
view_2  = CDSView(filter=IndexFilter([11,12,13,14,15]))                # Spring Valley
view_3  = CDSView(filter=IndexFilter([16,17]))                         # Carson Valley
view_4  = CDSView(filter=IndexFilter([18,19,20,21]))                   # Lower Colorado
view_5  = CDSView(filter=IndexFilter([22,23,24,25,26]))                # Oasis Valley
view_6  = CDSView(filter=IndexFilter([27,28,29,30,31,32,33,34,35]))    # Walker River
view_7  = CDSView(filter=IndexFilter([36,37]))                         # Bridgeport
view_8  = CDSView(filter=IndexFilter([38,39,40,41]))                   # Dixie Valley
view_9  = CDSView(filter=IndexFilter([42,43,44,45]))                   # Amargosa Desert
view_10 = CDSView(filter=IndexFilter([46,47,48,49,50,51]))             # Kobeh Valley
view_11 = CDSView(filter=IndexFilter([52,53]))                         # Amargosa Wild & Scenic River
view_12 = CDSView(filter=IndexFilter([54]))                            # Corn Creek
view_13 = CDSView(filter=IndexFilter([55,56,57,58]))                   # Lower Colorado, springs
view_14 = CDSView(filter=IndexFilter([59,60]))                         # Death Valley 


# create the figure
plot2 = figure(width=800, height=500, title="", tools="")

# best fit curve
plot2.line(x_new, ffit, line_width=1.5, line_color='#000000', legend_label="Best Fit Curve")


### data points
# BARCAS
br = plot2.scatter("VI", "ET_star", marker='diamond', size=11, color="black", source=source_ci, view=view_1)
# Spring Valley
sv = plot2.scatter("VI", "ET_star",  marker='square', size=6,  color="black", source=source_ci, view=view_2)
# Carson Valley
cv = plot2.scatter("VI", "ET_star", marker='triangle', size=8,  color="black", source=source_ci, view=view_3)
# Bridgeport
bp = plot2.scatter("VI", "ET_star", marker='circle',  size=6,  color="black", source=source_ci, view=view_7)
# Amargosa 
av = plot2.scatter("VI", "ET_star",   marker='hex',   size=8,  color="black", source=source_ci, view=view_9)
# Lower Colorado
lc = plot2.scatter("VI", "ET_star",  marker='diamond', size=11, color="black", source=source_ci, view=view_4,  fill_alpha=0)
# Oasis Valley
ov = plot2.scatter("VI", "ET_star",  marker='square', size=6,  color="black", source=source_ci, view=view_5,  fill_alpha=0)
# Walker River
wr = plot2.scatter("VI", "ET_star", marker='triangle', size=8,  color="black", source=source_ci, view=view_6,  fill_alpha=0)
# Dixie Valley
dv = plot2.scatter("VI", "ET_star",  marker='circle', size=6,  color="black", source=source_ci, view=view_8,  fill_alpha=0)
# Kobeh Valley 
kv = plot2.scatter("VI", "ET_star",   marker='hex',   size=7,  color="black", source=source_ci, view=view_10, fill_alpha=0)
# Amargosa
aw = plot2.scatter("VI", "ET_star", marker='inverted_triangle', size=8,  color="black", source=source_ci, view=view_11)
# Corn Creek
cc = plot2.scatter("VI", "ET_star", marker='star', size=8,  color="black", source=source_ci, view=view_12)
# Lower Colorado, springs
sp = plot2.scatter("VI", "ET_star", marker='plus', size=8,  color="black", source=source_ci, view=view_13)
# Lower Colorado, springs
dt = plot2.scatter("VI", "ET_star", marker='star', size=8,  color="black", source=source_ci, view=view_14, fill_alpha=0)

# 90% confidence and prediction bands
plot2.line(x_star, pi_l, line_width=1.3, line_color="black", line_dash='dashed', legend_label='90% PI')
plot2.line(x_star, pi_u, line_width=1.3, line_color="black", line_dash='dashed')
plot2.line(x_star, ci_l, line_width=1.3, line_color="black", line_dash='dotted', legend_label='90% CI')
plot2.line(x_star, ci_u, line_width=1.3, line_color="black", line_dash='dotted')

# hover tool attributes
hover_ci = HoverTool(
        tooltips=[
            ("Study", "@study"),
            ("Site", "@site"),
            ("Year", "@year"),
            (vi_type, "@VI{0.0000}"),
            ("ET*", "@ET_star{0.0000}")],
        renderers=[br, sv, cv, bp, av, lc, ov, wr, dv, kv, aw, cc, sp, dt]
    )
    
# add hover tool to the plot
plot2.add_tools(hover_ci)


### plot aesthetics

# title and legend
plot2.title.align = "center"
plot2.title.text_font_size = '16pt'
plot2.outline_line_width = 1
plot2.outline_line_color = "black"
plot2.legend.location = [40,300]
plot2.legend.label_text_font ='sans-serif'
plot2.legend.label_text_font_style ='normal'
plot2.legend.label_text_font_size = '10pt'
plot2.legend.border_line_color = None
plot2.legend.spacing = 1

# x axis
plot2.xgrid.grid_line_color = None
plot2.xaxis.axis_label = f"Source Area {vi_type}"
plot2.xaxis.axis_label_text_font = 'sans-serif'
plot2.xaxis.axis_label_text_font_style = 'normal'
plot2.xaxis.axis_label_text_font_size = '12pt'
plot2.xaxis.axis_label_standoff = 15
plot2.xaxis.major_tick_in = 6
plot2.xaxis.major_tick_out = 0
plot2.xaxis.major_tick_line_width = 1
plot2.xaxis.minor_tick_in = 4
plot2.xaxis.minor_tick_out = 0
plot2.x_range.start = 0
plot2.x_range.end = 0.9
plot2.xaxis[0].ticker.desired_num_ticks = 9

# y axis
plot2.yaxis.axis_label = "ET*"
plot2.yaxis.axis_label_text_font = 'sans-serif'
plot2.yaxis.axis_label_text_font_style = 'normal'
plot2.yaxis.axis_label_text_font_size = '12pt'
plot2.yaxis.axis_label_standoff = 20
plot2.ygrid.grid_line_color = None
plot2.yaxis.major_tick_in = 6
plot2.yaxis.major_tick_out = 0
plot2.yaxis.major_tick_line_width = 1
plot2.yaxis.minor_tick_in = 4
plot2.yaxis.minor_tick_out = 0
plot2.y_range.start = 0
plot2.y_range.end = 1.201

plot2.min_border_top = 65
plot2.min_border_bottom = 65
plot2.min_border_left = 65
plot2.min_border_right = 50

plot2.toolbar.logo = None
plot2.toolbar_location = None

if filter_outlier_flag:
    if wind_dir_flag:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered_constrained_confidence_intervals.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained_confidence_intervals.png" )
    else:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_outlier_filtered_constrained_confidence_intervals.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained_confidence_intervals.png")
else:
    if wind_dir_flag:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained_confidence_intervals.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_wind_dir_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained_confidence_intervals.png")
    else:
        out_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained_confidence_intervals.html")
        png_file_path = os.path.join(table_path, f"bmm_{vi_type.lower()}_et_{buffer}m_buffer_doy_{doy_st}_{doy_en}_{tiles_pr}_wrs2tiles_constrained_confidence_intervals.png")

output_file(out_file_path, mode='inline')
export_png(plot2, filename=png_file_path)
show(plot2)